# APEX-GRPO v3 on HotpotQA — Google Colab

Run setup and definition cells in order. Set the operation switches near the end only when ready.
Defaults: `RUN_PROFILE = "apex_v3"`, `SEED = 42`, `200` training episodes, `100` validation / `500` held-out test examples, Ollama `llama3.1:8b` execution LLM, Hugging Face `hotpotqa/hotpot_qa` distractor split, and `REWARD_MODE = "original_semantic"` (preserving 100% reward compatibility with `AutoDSPy.pdf`).

This **v3 notebook** unifies **all baseline, ablation, and adaptive runs** in a single confound-free codebase:
1. **Unified `RUN_PROFILE` Selector:** Switch cleanly between `apex_v3` (Ours), `static_k5_original` (AutoDSPy repro), `static_k5_matched`, `static_k12_matched`, and component ablations (`static_k5_stabilized`, `static_k5_debiased`, `ablation_no_prealloc`, `ablation_with_replacement`, `ablation_kmin5`) with zero code drift.
2. **Pillar 1 — Analytical Pre-Allocation (Layer B):** Exact 30-pipeline distribution via debiased logits ($z_\theta - b_0$) solves $\max_K [\mathbb{E}[U(K)] - \lambda K]$ ($\lambda = 0.25$) over $K \in \{2, 4, 8, 12\}$ with GVM-RAFT hopeless-prompt suppression.
3. **Pillar 2 — Intra-Episode Deduplication (Layer A):** Deterministic `temp = 0.0` executions run at most once per unique `(Module, Signature)` inside an episode.
4. **Pillar 3 — Active $K_\min=2$ Micro-Probe & Without-Replacement Rescue (Layer C):** Easy `[1, 1]` prompts exit immediately at $K=2$; failed groups expand through `(2, 4, 8, 12)` by drawing strictly untried pipelines with bounded importance weight $w_i = Z_{\mathcal{U}} \in (0, 1]$.
5. **Pillar 4 — Stabilized $1/K$ Update & Unified Contrast Masking (Layer D):** Enforces `REWARD_CONTRAST = 0.5` consistently in both early-exit and optimizer-step gating, adds a gentle exact 30-action entropy floor ($\lambda_H = 0.02$), and clips gradients (`max_norm = 1.0`).
6. **Multi-Model & Greedy Cached Evaluation:** Supports `gpt2` (fp32) and larger causal LMs (`Qwen/Qwen2.5-1.5B` with `bf16` + LoRA), greedy `argmax` validation/test decoding, an eval-only deterministic `eval_cache`, and built-in Zero-RL baselines (`DSPy-Predict`, `DSPy-CoT`, `Random`).


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

CACHE = Path("/content/drive/MyDrive/GRPO_cache")
CACHE.mkdir(parents=True, exist_ok=True)

# Reuse Python package downloads.
os.environ["PIP_CACHE_DIR"] = str(CACHE / "pip")

# Store Ollama models persistently.
# Must be set BEFORE starting "ollama serve".
os.environ["OLLAMA_MODELS"] = str(CACHE / "ollama_models")

In [ ]:
import os
import time
import subprocess
import urllib.request

print("=== Starting Colab Environment Setup ===")

# 1. Install system dependencies
print("Installing zstd...")
subprocess.run("sudo apt-get update -qq && sudo apt-get install -y zstd", shell=True, check=True)

# 2. Install Ollama
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 3. Start Ollama daemon in the background and wait until it is ready
print("Starting Ollama serve...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Poll the host endpoint until it responds
ready = False
for i in range(30):
    try:
        with urllib.request.urlopen("http://localhost:11434/") as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(2)

if ready:
    print("Ollama daemon is up and running!")
else:
    raise RuntimeError("Ollama daemon failed to start.")

# 4. Pull the model specified in your CONFIG
TARGET_MODEL = "llama3.1:8b"
print(f"Pulling Ollama model: {TARGET_MODEL} (this may take a moment)...")
check = subprocess.run(
    ["ollama", "show", TARGET_MODEL],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

if check.returncode != 0:
    subprocess.run(["ollama", "pull", TARGET_MODEL], check=True)
else:
    print("Using cached model:", TARGET_MODEL)
print("Setup complete and verified!")

In [ ]:
!ollama list

# 2. Environment and Dependency Setup
Use the existing GPU virtual environment as your Jupyter kernel. Installation is opt-in and does not replace your PyTorch build. Install CUDA-compatible PyTorch separately if needed. Version snapshots accompany every run; frozen versions alone do not guarantee Ollama determinism.

In [ ]:
%pip install -q dspy==3.3.0 transformers==5.14.1 datasets==5.0.1 "numpy>=2.0,<3" "matplotlib>=3.10,<4" "nbformat>=5.10" "peft>=0.13" ipykernel


The setup above follows the Static-K Colab notebook and downloads packages and `llama3.1:8b`. Keep the runtime active while Ollama runs. Drive stores cached models and backup copies; active training and validation run on local disk for speed.

# APEX-GRPO v3: HotpotQA (Unified Stabilized, Deduplicated & Adaptive GRPO)

This notebook implements **APEX-GRPO v3** for AutoDSPy pipeline synthesis:
- **Discrete 30-Pipeline Action Space:** Selects 1 of 2 DSPy modules (`CoT`, `Predict`) and 1 of 15 signatures, then `stop`.
- **Debiased Policy Parameterization:** Subtracts initial baseline logits $b_0$ at Episode 0 so exploration begins near an unbiased $1/30$ uniform prior instead of locking onto high-prior GPT-2 letter tokens (`" I"`).
- **Active Adaptive Sizing ($K \in \{2, 4, 8, 12\}$):** Mastered prompts exit at $K=2$; failed prompts expand up to $K_{\text{ceiling}}$ using **without-replacement rescue** with bounded IS correction $w_i = Z_{\mathcal{U}}$.
- **Contrast-Gated Masked $1/K$ Updates:** Preserves `original_semantic` judge fallback from `AutoDSPy.pdf` while skipping noisy fractional spreads $\le 0.5$ in both `adaptive_decision()` and `update_policy()`, plus mild entropy regularization ($\lambda_H = 0.02$) and gradient clipping (`max_norm = 1.0`).

**Saved outputs follow immutable protocols. Restart kernel and rerun definition cells in order.**


# 2. Environment and Dependency Setup
Use the existing GPU virtual environment as your Jupyter kernel. Installation is opt-in and does not replace your PyTorch build. Install CUDA-compatible PyTorch separately if needed. Version snapshots accompany every run; frozen versions alone do not guarantee Ollama determinism.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dspy==3.3.0", "transformers==5.14.1",
                           "datasets==5.0.1", "numpy>=2.0,<3", "matplotlib>=3.10,<4", "nbformat>=5.10",
                           "peft>=0.13", "ipykernel"])

import collections, copy, gc, hashlib, importlib.metadata, json, math, os, random, re, string, sys, tempfile, time, uuid
from contextlib import contextmanager
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np
import torch
import dspy
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, GPT2LMHeadModel, GPT2Tokenizer

VERSIONS = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "dspy", "datasets", "numpy", "matplotlib"]}
print("Python:", sys.version, "\nPackages:", VERSIONS, "\nCUDA:", torch.cuda.is_available())


# 3. Experiment Configuration
Edit **only** the `# --- PRIMARY USER CONTROLS ---` block at the top of the next cell (`RUN_PROFILE`, `SEED`, `PRESET`, `POLICY_MODEL`, `USE_LORA`, `LEARNING_RATE`, `OLLAMA_MODEL`), then rerun that cell to rebuild `CONFIG`.

- Keep `REWARD_MODE = "original_semantic"` for 100% reward compatibility with `AutoDSPy.pdf`.
- Use `RUN_PROFILE` to select between `apex_v3` (Ours), `static_k5_original`, `static_k5_matched`, `static_k12_matched`, or any of the 5 component ablation presets (`static_k5_stabilized`, `static_k5_debiased`, `ablation_no_prealloc`, `ablation_with_replacement`, `ablation_kmin5`).
- Set `USE_LORA = True` and `LEARNING_RATE = 5e-5` when training a 1B–1.5B policy model (e.g., `Qwen/Qwen2.5-1.5B`) on a 24 GB Colab L4 alongside `llama3.1:8b`.


In [ ]:
# --- PRIMARY USER CONTROLS (Edit ONLY this block per run/seed) ---
RUN_PROFILE = "apex_v3"                  # See RUN_PROFILES below for all 9 presets
SEED = 42                                # 42, 43, 44
PRESET = "baseline"                      # "smoke" (5 ep) | "pilot" (10 ep) | "baseline" (200 ep)
POLICY_MODEL = "gpt2"                    # "gpt2" | "Qwen/Qwen2.5-1.5B" | "meta-llama/Llama-3.2-1B"
POLICY_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"  # Auto-cleared to None when POLICY_MODEL != "gpt2"
USE_LORA = False                         # Set True for Qwen/Qwen2.5-1.5B on 24 GB Colab L4
LEARNING_RATE = 1e-5                     # Use 1e-5 for GPT-2, 5e-5 when USE_LORA = True
OLLAMA_MODEL = "llama3.1:8b"             # "llama3.1:8b" | "llama3.2:3b"
REWARD_MODE = "original_semantic"       # original_semantic / answer_exact
VALIDATION_SIZE = 100
TEST_SIZE = 500
EVAL_DECODING = "greedy"                 # "greedy" (argmax) | "multinomial"
EVAL_CACHE_ENABLED = True                # Eval-only deterministic cache (temp=0.0); never used in training
# ----------------------------------------------------------------------

JUDGE_CACHE_ENABLED = True
JUDGE_LOGGING_ENABLED = True
LEGACY_REWARD_PROTOCOL = False

MODE = "initialize"                      # initialize / train / resume / validate / test / compare
NUM_EPISODES = None                      # None chooses preset; override freely
TRAIN_POOL_SIZE = 200
OLLAMA_HOST = "http://localhost:11434"
CHECKPOINT_INTERVAL = 10
VALIDATION_INTERVAL = None               # baseline 50; smoke 5; pilot 10
SCHEDULE_EPISODES = 200                  # common maximum; presets use prefixes
DATASET_REVISION = None                  # resolve main to immutable HF commit before loading
POLICY_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OLLAMA_NUM_GPU = None                    # 0=CPU; None=Ollama default; integer=GPU layers
OLLAMA_NUM_CTX = 4096
EXECUTION_MAX_TOKENS = 512
EXECUTION_TEMPERATURE = 0.0
REQUEST_TIMEOUT = 180
RESULTS_ROOT = Path("/content/adaptive_grpo_results")
SELECTED_CHECKPOINT = None               # explicit path for resume / validate / test
RUN_OPERATION = False                    # section 21 safety switch
RUN_ALL_K = False                        # unused in v3; retained for schema parity
ALLOWED_K = (2, 4, 5, 8, 10, 12, 15)
PROTOCOL_VERSION = "hotpotqa-apex-adaptive-rollout-v3"

JUDGE_MODEL = None
JUDGE_MODEL_DIGEST = None
JUDGE_TEMPERATURE = 0.0
JUDGE_SEED = None
JUDGE_MAX_TOKENS = 512
JUDGE_NUM_CTX = 4096
JUDGE_TOP_P = 1.0
JUDGE_TOP_K = 40
JUDGE_REPEAT_PENALTY = 1.1
JUDGE_MAX_ATTEMPTS = 1
JUDGE_RETRY_DELAY_SECONDS = 0.0
JUDGE_CACHE_POLICY = "run_private"
JUDGE_CACHE_ROOT = None
JUDGE_CACHE_SNAPSHOT = None

# APEX-GRPO v3 Adaptive Rollout & Ablation Configuration
CANDIDATE_BUDGET = None
RESCUE_MODE = "failure_only"
LAMBDA_PENALTY = 0.25
ALLOCATION_VERSION = "apex-adaptive-rollout-v3"
GPT2_PINNED_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"

RUN_PROFILES = {
    "apex_v3": dict(
        adaptive_levels=(2, 4, 8, 12), use_adaptive_k=True, use_prealloc=True,
        use_without_replacement=True, rescue_is_correction=True, use_dedup_cache=True,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="unique-signature-token-label-v1", debiased_logits=True,
        entropy_coef=0.02, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
    "static_k5_original": dict(
        adaptive_levels=(5,), use_adaptive_k=False, use_prealloc=False,
        use_without_replacement=False, rescue_is_correction=False, use_dedup_cache=False,
        normalize_loss_by_k=False, use_zero_var_mask=False,
        policy_sampling="restricted-first-token-multinomial", debiased_logits=False,
        entropy_coef=0.0, grad_clip_norm=None, reward_contrast=0.0,
        failure_reward_threshold=0.0, rescue_mode="failure_only",
    ),
    "static_k5_stabilized": dict(
        adaptive_levels=(5,), use_adaptive_k=False, use_prealloc=False,
        use_without_replacement=False, rescue_is_correction=False, use_dedup_cache=False,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="restricted-first-token-multinomial", debiased_logits=False,
        entropy_coef=0.0, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
    "static_k5_debiased": dict(
        adaptive_levels=(5,), use_adaptive_k=False, use_prealloc=False,
        use_without_replacement=False, rescue_is_correction=False, use_dedup_cache=False,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="unique-signature-token-label-v1", debiased_logits=True,
        entropy_coef=0.02, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
    "static_k5_matched": dict(
        adaptive_levels=(5,), use_adaptive_k=False, use_prealloc=False,
        use_without_replacement=False, rescue_is_correction=False, use_dedup_cache=True,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="unique-signature-token-label-v1", debiased_logits=True,
        entropy_coef=0.02, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
    "static_k12_matched": dict(
        adaptive_levels=(12,), use_adaptive_k=False, use_prealloc=False,
        use_without_replacement=False, rescue_is_correction=False, use_dedup_cache=True,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="unique-signature-token-label-v1", debiased_logits=True,
        entropy_coef=0.02, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
    "ablation_no_prealloc": dict(
        adaptive_levels=(2, 4, 8, 12), use_adaptive_k=True, use_prealloc=False,
        use_without_replacement=True, rescue_is_correction=True, use_dedup_cache=True,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="unique-signature-token-label-v1", debiased_logits=True,
        entropy_coef=0.02, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
    "ablation_with_replacement": dict(
        adaptive_levels=(2, 4, 8, 12), use_adaptive_k=True, use_prealloc=True,
        use_without_replacement=False, rescue_is_correction=False, use_dedup_cache=True,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="unique-signature-token-label-v1", debiased_logits=True,
        entropy_coef=0.02, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
    "ablation_kmin5": dict(
        adaptive_levels=(5, 8, 10, 12), use_adaptive_k=True, use_prealloc=True,
        use_without_replacement=True, rescue_is_correction=True, use_dedup_cache=True,
        normalize_loss_by_k=True, use_zero_var_mask=True,
        policy_sampling="unique-signature-token-label-v1", debiased_logits=True,
        entropy_coef=0.02, grad_clip_norm=1.0, reward_contrast=0.5,
        failure_reward_threshold=0.25, rescue_mode="failure_only",
    ),
}

REWARD_MODES = ("original_semantic", "answer_exact")
REWARD_SETTING_KEYS = {"reward_mode", "semantic_fallback"}

def effective_reward_config(cfg):
    mode = cfg.get("reward_mode", "original_semantic")
    return dict(reward_mode=mode, reward_protocol_version="hotpotqa-original-semantic-v1" if mode=="original_semantic" else "answer-exact-v1",
                semantic_fallback=(mode != "answer_exact"))

def reward_protocol_identity(cfg):
    return effective_reward_config(cfg)

def canonical_experiment_config(cfg):
    return dict(config={k:copy.deepcopy(v) for k,v in cfg.items() if k not in REWARD_SETTING_KEYS},
                reward=reward_protocol_identity(cfg))

def build_config(**overrides):
    presets = {"smoke": 5, "pilot": 10, "baseline": 200}
    preset = overrides.get("preset", PRESET)
    if preset not in presets:
        raise ValueError("PRESET must be smoke, pilot or baseline")
    profile_name = overrides.get("run_profile", RUN_PROFILE)
    if profile_name not in RUN_PROFILES:
        raise ValueError(f"RUN_PROFILE must be one of {sorted(RUN_PROFILES)}")
    profile = copy.deepcopy(RUN_PROFILES[profile_name])
    policy_model = overrides.get("policy_model", POLICY_MODEL)
    if "policy_revision" in overrides:
        policy_revision = overrides["policy_revision"]
    elif policy_model != "gpt2" and POLICY_REVISION == GPT2_PINNED_REVISION:
        policy_revision = None
    else:
        policy_revision = POLICY_REVISION

    val_int = VALIDATION_INTERVAL if VALIDATION_INTERVAL is not None else {"smoke": 5, "pilot": 10, "baseline": 50}[preset]
    cfg = dict(
        dataset="hotpotqa",
        dataset_name="hotpotqa/hotpot_qa",
        dataset_config="distractor",
        dataset_revision=DATASET_REVISION,
        run_profile=profile_name,
        policy_model=policy_model,
        policy_revision=policy_revision,
        use_lora=USE_LORA,
        k=max(profile["adaptive_levels"]),
        seed=SEED,
        preset=preset,
        num_episodes=NUM_EPISODES if NUM_EPISODES is not None else presets[preset],
        train_pool_size=TRAIN_POOL_SIZE,
        validation_size=VALIDATION_SIZE,
        test_size=TEST_SIZE,
        schedule_episodes=SCHEDULE_EPISODES,
        learning_rate=LEARNING_RATE,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        validation_interval=val_int,
        policy_device=POLICY_DEVICE,
        ollama_model=OLLAMA_MODEL,
        ollama_host=OLLAMA_HOST.rstrip("/"),
        ollama_num_gpu=OLLAMA_NUM_GPU,
        ollama_num_ctx=OLLAMA_NUM_CTX,
        execution_max_tokens=EXECUTION_MAX_TOKENS,
        execution_temperature=EXECUTION_TEMPERATURE,
        request_timeout=REQUEST_TIMEOUT,
        results_root=str(RESULTS_ROOT),
        protocol=PROTOCOL_VERSION,
        protocol_version=PROTOCOL_VERSION,
        policy_dropout=False,
        loss="negative-normalized-logprob-times-centered-reward",
        optimizer="Adam",
        cache=False,
        semantic_fallback=True,
        max_policy_tokens=1024,
        max_pipeline_steps=3,
        evaluation_sampling="greedy-argmax" if EVAL_DECODING == "greedy" else "multinomial-with-fixed-split-seed",
        eval_decoding=EVAL_DECODING,
        eval_cache_enabled=EVAL_CACHE_ENABLED,
        evaluation_seed_offset=100000,
        candidate_budget=CANDIDATE_BUDGET,
        lambda_penalty=LAMBDA_PENALTY,
        allocation_version=ALLOCATION_VERSION,
    )
    cfg.update(profile)
    cfg["adaptive_levels"] = list(cfg["adaptive_levels"])
    cfg.update(
        reward_mode=REWARD_MODE, legacy_reward_protocol=LEGACY_REWARD_PROTOCOL,
        judge_cache_enabled=False if LEGACY_REWARD_PROTOCOL else JUDGE_CACHE_ENABLED,
        judge_logging_enabled=JUDGE_LOGGING_ENABLED, judge_model=JUDGE_MODEL, judge_model_digest=JUDGE_MODEL_DIGEST,
        judge_temperature=JUDGE_TEMPERATURE, judge_seed=JUDGE_SEED, judge_max_tokens=JUDGE_MAX_TOKENS,
        judge_num_ctx=JUDGE_NUM_CTX, judge_top_p=JUDGE_TOP_P, judge_top_k=JUDGE_TOP_K,
        judge_repeat_penalty=JUDGE_REPEAT_PENALTY, judge_max_attempts=JUDGE_MAX_ATTEMPTS,
        judge_retry_delay_seconds=JUDGE_RETRY_DELAY_SECONDS, judge_cache_policy=JUDGE_CACHE_POLICY,
        judge_cache_root=JUDGE_CACHE_ROOT, judge_cache_snapshot=JUDGE_CACHE_SNAPSHOT,
    )
    cfg.update(overrides)
    cfg["adaptive_levels"] = list(cfg["adaptive_levels"])
    cfg["k"] = max(cfg["adaptive_levels"])
    if "eval_decoding" in overrides and "evaluation_sampling" not in overrides:
        cfg["evaluation_sampling"] = "greedy-argmax" if cfg["eval_decoding"] == "greedy" else "multinomial-with-fixed-split-seed"
    if cfg.get("legacy_reward_protocol") and "judge_cache_enabled" not in overrides:
        cfg["judge_cache_enabled"] = False
    if cfg["k"] not in ALLOWED_K:
        raise ValueError(f"K must be one of {ALLOWED_K}, got {cfg['k']}")
    for key in ["num_episodes", "train_pool_size", "validation_size", "test_size", "schedule_episodes",
                "checkpoint_interval", "validation_interval", "execution_max_tokens", "ollama_num_ctx"]:
        if not isinstance(cfg[key], int) or cfg[key] <= 0:
            raise ValueError(f"{key} must be a positive integer")
    cfg["schedule_episodes"] = max(cfg["schedule_episodes"], cfg["num_episodes"])
    if cfg["learning_rate"] <= 0 or cfg["policy_device"] not in ("cpu", "cuda"):
        raise ValueError("Use positive learning rate and cpu/cuda policy_device")
    if cfg["eval_decoding"] not in ("greedy", "multinomial"):
        raise ValueError("eval_decoding must be greedy or multinomial")
    if cfg["candidate_budget"] is not None and (type(cfg["candidate_budget"]) is not int or cfg["candidate_budget"] < cfg["adaptive_levels"][0]):
        raise ValueError("candidate budget must be None or at least the initial group size")
    levels = cfg["adaptive_levels"]
    if (len(levels) < 1 or any(type(x) is not int or x < 2 for x in levels)
            or levels != sorted(set(levels)) or levels[-1] > cfg["k"]):
        raise ValueError("Adaptive levels must be increasing integers in [2, K]")
    if cfg["rescue_mode"] not in ("both", "failure_only"):
        raise ValueError("rescue mode must be both or failure_only")
    if not math.isfinite(cfg["reward_contrast"]) or not 0 <= cfg["reward_contrast"] < 1:
        raise ValueError("reward contrast must be in [0,1)")
    if not math.isfinite(cfg["failure_reward_threshold"]) or not 0 <= cfg["failure_reward_threshold"] < 1:
        raise ValueError("failure_reward_threshold must be in [0,1)")
    if not math.isfinite(cfg["lambda_penalty"]) or cfg["lambda_penalty"] <= 0:
        raise ValueError("lambda_penalty must be a positive finite float")
    if not math.isfinite(cfg["entropy_coef"]) or cfg["entropy_coef"] < 0:
        raise ValueError("entropy_coef must be non-negative and finite")
    if cfg["grad_clip_norm"] is not None and (not math.isfinite(cfg["grad_clip_norm"]) or cfg["grad_clip_norm"] <= 0):
        raise ValueError("grad_clip_norm must be None or positive finite float")
    if cfg["reward_mode"] not in REWARD_MODES:
        raise ValueError(f"reward_mode must be one of {REWARD_MODES}")
    cfg["semantic_fallback"] = (cfg["reward_mode"] != "answer_exact")
    return cfg

def adaptive_decision(rewards, failures, cfg, k_ceiling=None):
    """
    Pillar 3: Evaluates staged stopping decisions at level boundaries.
    Enforces static short-circuit, contrast threshold (> reward_contrast),
    All-Correct Invariant, failure threshold (<= failure_reward_threshold), and ceiling cap.
    """
    current_k = len(rewards)
    levels = cfg["adaptive_levels"]
    if current_k not in levels:
        return None
    if failures:
        return "failure_guard"
    if not cfg.get("use_adaptive_k", True):
        return "static_fixed_k"
    values = np.asarray(rewards, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite reward")

    # 1. Reward contrast observed above contrast threshold -> exit immediately
    if float(values.max() - values.min()) > cfg["reward_contrast"]:
        return "contrast"

    # 2. All-correct invariant: all candidates succeeded -> exit immediately
    if float(values.min()) == 1.0:
        return "all_correct_exit"

    # 3. Non-failure homogeneous (e.g., all 0.5-0.9 under fractional judge scores)
    if cfg.get("rescue_mode") == "failure_only" and float(values.max()) > cfg.get("failure_reward_threshold", 0.25):
        return "non_failure_homogeneous"

    # 4. Pre-allocation ceiling and maximum checks
    effective_ceiling = k_ceiling if (k_ceiling is not None and cfg.get("use_prealloc", True)) else levels[-1]
    if current_k >= effective_ceiling:
        return "ceiling_reached"
    if current_k == levels[-1]:
        return "maximum_k"
    return None


STATIC_RUN_CONFIG_PATH = None
STATIC_REFERENCE = None
if STATIC_RUN_CONFIG_PATH:
    STATIC_REFERENCE = json.loads(Path(STATIC_RUN_CONFIG_PATH).read_text(encoding="utf-8"))
    reference_config = STATIC_REFERENCE["config"]
    required = ("dataset_revision", "policy_revision", "policy_model", "ollama_model", "seed",
                "train_pool_size", "validation_size", "test_size", "schedule_episodes")
    if any(reference_config.get(key) is None for key in required):
        raise ValueError("Static run config is missing a resolved dataset, model, or selection setting")
    DATASET_REVISION = reference_config["dataset_revision"]
    POLICY_REVISION = reference_config["policy_revision"]
    POLICY_MODEL = reference_config["policy_model"]
    OLLAMA_MODEL = reference_config["ollama_model"]
    SEED = reference_config["seed"]
    TRAIN_POOL_SIZE = reference_config["train_pool_size"]
    VALIDATION_SIZE = reference_config["validation_size"]
    TEST_SIZE = reference_config["test_size"]
    SCHEDULE_EPISODES = reference_config["schedule_episodes"]
    print("Pinned reference dataset/model settings from", STATIC_RUN_CONFIG_PATH)

CONFIG = build_config()
print("Active Profile:", CONFIG["run_profile"], "| Levels:", CONFIG["adaptive_levels"],
      "| Policy:", CONFIG["policy_model"], "| LoRA:", CONFIG["use_lora"],
      "| Exec:", CONFIG["ollama_model"], "| Seed:", CONFIG["seed"])


### Drive backup, auto-namespacing, and eval-only cache

Checkpoints and their logs are copied to an auto-namespaced folder under `MyDrive/APEX_GRPO_v3/<policy>_<executor>/<dataset>/<RUN_PROFILE>/seed<SEED>` after every checkpoint and again after tests and plots so runs never overwrite one another.

An **evaluation-only** deterministic cache (`_eval_cache/`) stores `temp = 0.0` validation/test pipeline evaluations by exact `(dataset, model_digest, reward_protocol, question_id, module, signature)` so repeated evaluations across checkpoints and baselines reuse identical deterministic calls. **Training never reads `_eval_cache/`** (training uses only intra-episode deduplication).


In [ ]:
# Colab Drive backup controls & eval-only deterministic cache.
from pathlib import Path
import shutil
import json
from datetime import datetime, timezone

BACKUP_TO_DRIVE = True
RESTORE_FROM_DRIVE = False  # Set True only in a fresh runtime before resuming.

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

def resolve_drive_backup_dir(cfg):
    policy_short = cfg["policy_model"].split("/")[-1].replace(":", "-")
    exec_short = cfg["ollama_model"].replace(":", "-")
    return Path(f"/content/drive/MyDrive/APEX_GRPO_v3/{policy_short}_{exec_short}/{cfg['dataset']}/{cfg['run_profile']}/seed{cfg['seed']}")

def resolve_eval_cache_path(cfg):
    exec_short = cfg["ollama_model"].replace(":", "-")
    base = Path("/content/drive/MyDrive/APEX_GRPO_v3/_eval_cache") if (IN_COLAB and BACKUP_TO_DRIVE) else (Path(cfg["results_root"]) / "_eval_cache")
    return base / f"{cfg['dataset']}_{exec_short}_{cfg['reward_mode']}.jsonl"

DRIVE_BACKUP_DIR = resolve_drive_backup_dir(CONFIG)
print("Resolved Drive Backup Dir:", DRIVE_BACKUP_DIR)

class DeterministicEvalCache:
    """Append-only evaluation cache (temp=0.0 only). Strictly forbidden inside train_grpo."""
    def __init__(self, path):
        self.path = Path(path)
        self.entries = {}
        if self.path.is_file():
            for line in self.path.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                    self.entries[rec["key"]] = (rec["execution"], rec["reward"])
                except Exception:
                    continue

    @staticmethod
    def make_key(cfg, llm_metadata, question_id, module, signature):
        digest = (llm_metadata or {}).get("model", {}).get("digest", "unknown")
        payload = dict(
            dataset=cfg["dataset"],
            ollama_model=cfg["ollama_model"],
            model_digest=digest,
            execution_temperature=cfg["execution_temperature"],
            execution_max_tokens=cfg["execution_max_tokens"],
            ollama_num_ctx=cfg["ollama_num_ctx"],
            reward_mode=cfg["reward_mode"],
            judge_model_digest=cfg.get("judge_model_digest") or digest,
            question_id=str(question_id),
            module=str(module),
            signature=str(signature),
        )
        return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

    def get(self, key):
        hit = self.entries.get(key)
        if hit is None:
            return None
        return copy.deepcopy(hit[0]), copy.deepcopy(hit[1])

    def put(self, key, execution, reward):
        if key in self.entries:
            return
        ex_copy = copy.deepcopy(execution)
        rw_copy = copy.deepcopy(reward)
        self.entries[key] = (ex_copy, rw_copy)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        record = dict(key=key, execution=ex_copy, reward=rw_copy)
        with self.path.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
            stream.flush()

def copy_backup_file(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".uploading")
    shutil.copyfile(source, temporary)
    temporary.replace(destination)

def backup_results(cfg=None):
    if not IN_COLAB or not BACKUP_TO_DRIVE:
        return
    local = RESULTS_ROOT.resolve()
    if cfg is not None and Path(cfg["results_root"]).resolve() != local:
        return
    if not local.is_dir():
        return
    target_dir = resolve_drive_backup_dir(cfg) if cfg is not None else DRIVE_BACKUP_DIR
    destination = target_dir / "results"
    remote = destination.resolve()
    if local == remote or local in remote.parents or remote in local.parents:
        raise ValueError("Keep local RESULTS_ROOT separate from the Drive backup")
    target_dir.mkdir(parents=True, exist_ok=True)
    marker = target_dir / "backup_complete.json"
    marker.unlink(missing_ok=True)
    files = {}
    for item in sorted(local.rglob("*")):
        if not item.is_file() or item.name.endswith((".tmp", ".uploading")) or "_eval_cache" in item.parts:
            continue
        relative = item.relative_to(local)
        target = destination / relative
        size = item.stat().st_size
        if item.suffix != ".pt" or not target.exists() or target.stat().st_size != size:
            copy_backup_file(item, target)
        files[relative.as_posix()] = size
    metadata = dict(local_root=str(local), files=files,
                    saved_at=datetime.now(timezone.utc).isoformat())
    temporary = marker.with_suffix(".uploading")
    temporary.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    temporary.replace(marker)
    print("Drive backup complete:", target_dir)

def restore_results(cfg=None):
    if not IN_COLAB:
        raise RuntimeError("Run Drive recovery inside Colab")
    target_dir = resolve_drive_backup_dir(cfg) if cfg is not None else DRIVE_BACKUP_DIR
    marker = target_dir / "backup_complete.json"
    if not marker.is_file():
        raise RuntimeError(f"No complete backup found at {target_dir}; check RUN_PROFILE/SEED/model")
    metadata = json.loads(marker.read_text(encoding="utf-8"))
    local = RESULTS_ROOT.resolve()
    if str(local) != metadata["local_root"]:
        raise ValueError("Use the original working directory and RESULTS_ROOT: " + metadata["local_root"])
    if local.exists() and any(local.iterdir()):
        raise FileExistsError("Restore only into an empty results folder; existing results are protected")
    remote = target_dir / "results"
    for relative, size in metadata["files"].items():
        item, target = remote / relative, local / relative
        if not target.resolve().is_relative_to(local) or not item.resolve().is_relative_to(remote.resolve()):
            raise ValueError("Invalid backup path")
        if not item.is_file() or item.stat().st_size != size:
            raise RuntimeError("Incomplete backup file: " + relative)
    for relative in metadata["files"]:
        copy_backup_file(remote / relative, local / relative)
    print("Restored local results from:", target_dir, "->", local)

if RESTORE_FROM_DRIVE:
    restore_results(CONFIG)


# 4. Hardware and Ollama Initialization
No server is launched or model pulled automatically. Start Ollama yourself and run `ollama pull llama3.1:8b` if absent. The endpoint is configurable, including a remote machine.

The instrumented original `dspy.LM('ollama_chat/...')` counts LM **forward attempts**, including adapter retries and semantic judges; these are logical calls. Exact HTTP request counts are unavailable because provider internals are not instrumented. Cache is disabled and provider retries are zero. Returned token usage is measured when present; failures may have unreported tokens. Ollama `/api/ps` reports server model residency, distinct from PyTorch allocator memory.

In [ ]:
def json_request(host, route, payload=None, timeout=15):
    data = None if payload is None else json.dumps(payload).encode()
    request = Request(host.rstrip("/")+route, data=data, headers={"Content-Type":"application/json"})
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)

def ollama_snapshot(cfg):
    try:
        return json_request(cfg["ollama_host"], "/api/ps", timeout=5)
    except Exception as exc:
        return {"unavailable": f"{type(exc).__name__}: {exc}"}

class MeasuredLM(dspy.LM):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.events = []
    def forward(self, *args, **kwargs):
        event = {"seconds": None, "prompt_tokens": None, "generated_tokens": None, "error": None}
        started = time.perf_counter()
        try:
            response = super().forward(*args, **kwargs)
            usage = dict(getattr(response, "usage", {}) or {})
            event["prompt_tokens"] = usage.get("prompt_tokens")
            event["generated_tokens"] = usage.get("completion_tokens")
            return response
        except Exception as exc:
            event["error"] = f"{type(exc).__name__}: {exc}"
            raise
        finally:
            event["seconds"] = time.perf_counter()-started
            self.events.append(event)

def initialize_ollama(cfg):
    try:
        version = json_request(cfg["ollama_host"], "/api/version")
        tags = json_request(cfg["ollama_host"], "/api/tags")["models"]
        model = next((m for m in tags if m.get("name")==cfg["ollama_model"] or m.get("model")==cfg["ollama_model"]), None)
        if model is None:
            raise RuntimeError(f"Missing {cfg['ollama_model']}; run ollama pull {cfg['ollama_model']}")
        show = json_request(cfg["ollama_host"], "/api/show", {"model":cfg["ollama_model"]})
    except Exception as exc:
        raise RuntimeError(f"Ollama unavailable at {cfg['ollama_host']}: {exc}") from exc
    options = dict(api_base=cfg["ollama_host"], api_key="", cache=False, num_retries=0,
                   temperature=cfg["execution_temperature"], max_tokens=cfg["execution_max_tokens"],
                   seed=cfg["seed"], num_ctx=cfg["ollama_num_ctx"], timeout=cfg["request_timeout"], keep_alive="5m")
    if cfg["ollama_num_gpu"] is not None:
        options["num_gpu"] = cfg["ollama_num_gpu"]
    lm = MeasuredLM("ollama_chat/"+cfg["ollama_model"], **options)
    dspy.configure(lm=lm)
    metadata = {"version":version, "model":model, "details":show.get("details"),
                "model_info":show.get("model_info"), "options":options, "ps":ollama_snapshot(cfg)}
    return lm, metadata

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng(states):
    random.setstate(states["python"]); np.random.set_state(states["numpy"]); torch.set_rng_state(states["torch"])
    if states["cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("Checkpoint has CUDA RNG state; resume in its original GPU environment")
        torch.cuda.set_rng_state_all(states["cuda"])

@contextmanager
def isolated_rng(seed):
    saved = rng_state()
    seed_everything(seed)
    try:
        yield
    finally:
        restore_rng(saved)

def hardware_metadata():
    return dict(python=sys.version, packages=VERSIONS, cuda_available=torch.cuda.is_available(),
                cuda_version=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)

# 5. Dataset Loading
Official [HotpotQA](https://huggingface.co/datasets/hotpotqa/hotpot_qa), `distractor` setting. The dataset commit is resolved and pinned on first load; rerun with an explicit commit to reproduce a historical download.

In [ ]:
def load_hotpotqa(cfg):
    cfg = copy.deepcopy(cfg)
    try:
        if cfg["dataset_revision"] is None:
            cfg["dataset_revision"] = HfApi().dataset_info(cfg["dataset_name"]).sha
        # These are labeled splits. No unlabeled official test is loaded or scored.
        datasets = load_dataset(cfg["dataset_name"], cfg["dataset_config"],
                                revision=cfg["dataset_revision"])
        if not {"train", "validation"}.issubset(datasets):
            raise ValueError("Required labeled HotpotQA splits unavailable")
    except Exception as exc:
        raise RuntimeError("Cannot load official HotpotQA distractor; check network/cache/revision: "+str(exc)) from exc
    return cfg, datasets

def format_hotpot_input(question, context):
    """Use only question and ALL provided paragraphs, in original order."""
    titles, paragraphs = context["title"], context["sentences"]
    if len(titles) != len(paragraphs):
        raise ValueError("Malformed HotpotQA context: mismatched titles/paragraphs")
    sections = [f"Title: {title}\n{''.join(sentences)}" for title, sentences in zip(titles, paragraphs)]
    return "Context:\n"+"\n\n".join(sections)+"\n\nQuestion: "+str(question)

def prepare_hotpot_example(item, split, index):
    if not isinstance(item.get("answer"), str) or not item["answer"].strip():
        raise ValueError("Labeled HotpotQA answer unavailable; refuse fabricated ground truth")
    context = copy.deepcopy(item["context"])
    return dict(id=str(item["id"]), split=split, index=int(index), question=item["question"],
                answer=item["answer"], input=format_hotpot_input(item["question"], context),
                context=context, metadata={key:copy.deepcopy(item.get(key))
                    for key in ("type", "level", "supporting_facts")})

def dataset_example(dataset, split, index):
    return prepare_hotpot_example(dataset[split][int(index)], split, index)

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value, overwrite=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        raise FileExistsError(f"Refusing overwrite: {path}")
    tmp = path.with_name(path.name+"."+uuid.uuid4().hex+".tmp")
    with tmp.open("w",encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.flush(); os.fsync(stream.fileno())
    if overwrite:
        os.replace(tmp,path)
    else:
        os.link(tmp,path)
        tmp.unlink()


# 6. Dataset Sampling and Episode Schedule
A dataset/seed/size/revision manifest is shared across K and presets. Training and validation are disjoint subsets of the labeled training split; held-out testing is sampled from the labeled validation split. Episodes draw with replacement from the fixed training pool.


In [ ]:
def selection_payload(cfg, datasets):
    ntrain, ntest = len(datasets["train"]), len(datasets["validation"])
    if cfg["train_pool_size"]+cfg["validation_size"] > ntrain or cfg["test_size"] > ntest:
        raise ValueError("Requested subset exceeds official split; reduce configured sizes")
    permutation = np.random.default_rng(cfg["seed"]).permutation(ntrain)
    train_idx = permutation[:cfg["train_pool_size"]].tolist()
    val_idx = permutation[cfg["train_pool_size"]:cfg["train_pool_size"]+cfg["validation_size"]].tolist()
    test_idx = np.random.default_rng(cfg["seed"]+1).permutation(ntest)[:cfg["test_size"]].tolist()
    subsets = {"train":[dataset_example(datasets,"train",i) for i in train_idx],
               "validation":[dataset_example(datasets,"train",i) for i in val_idx],
               "test":[dataset_example(datasets,"validation",i) for i in test_idx]}
    schedule = np.random.default_rng(cfg["seed"]+2).integers(0,cfg["train_pool_size"],size=cfg["schedule_episodes"]).tolist()
    identity = {key:cfg[key] for key in ["dataset_name","dataset_config","dataset_revision","seed",
                                        "train_pool_size","validation_size","test_size","schedule_episodes"]}
    return dict(identity=identity, indices=dict(train=train_idx,validation=val_idx,test=test_idx),
                ids={name:[item["id"] for item in rows] for name,rows in subsets.items()},
                content_hashes={name:stable_hash(rows) for name,rows in subsets.items()},
                episode_pool_offsets=schedule,
                episode_question_ids=[subsets["train"][offset]["id"] for offset in schedule]), subsets

def prepare_selection(cfg, datasets):
    payload, subsets = selection_payload(cfg,datasets)
    selection_id = stable_hash(payload)
    path = Path(cfg["results_root"])/"hotpotqa"/"selections"/(stable_hash(payload["identity"])+".json")
    if path.exists():
        persisted = json.loads(path.read_text(encoding="utf-8"))
        if persisted != payload:
            raise RuntimeError("Persisted dataset selection/schedule mismatch; do not overwrite it")
    else:
        write_json(path,payload)
    assert set(payload["indices"]["train"]).isdisjoint(payload["indices"]["validation"])
    assert set(payload["ids"]["train"]).isdisjoint(payload["ids"]["test"])
    assert set(payload["ids"]["validation"]).isdisjoint(payload["ids"]["test"])
    return dict(subsets=subsets, selection=payload, selection_id=selection_id, selection_path=str(path),
                full_test=datasets["validation"])

# Repeated draws are allowed, not forced. Small schedules may happen to contain no duplicate.
# Schedule identity never depends on K or preset; episodes use its leading prefix.

# 7. DSPy Action Space and Signatures
The released module pool and all fifteen signatures are retained, including signatures that do not naturally fit math QA. The policy selects one module, one signature, then deterministic `stop`; this released toy implementation therefore executes one DSPy stage per pipeline. No ReAct, retrieval, teleprompter, or new action head.

In [ ]:
MODULES = ["CoT", "Predict"]
SIGNATURES = ["question -> answer", "text -> summary", "question -> reasoning", "question -> hypothesis",
              "problem -> solution", "problem_description -> explanation", "context -> summary", "context -> briefing",
              "word_problem -> solution", "math_problem -> answer", "prompt -> response", "query -> response",
              "text -> response", "prompt -> generated_text", "query -> generated_text"]
ACTIONS = MODULES + SIGNATURES + ["stop"]

def valid_actions(partial):
    return MODULES if not partial else SIGNATURES if len(partial)==1 else ["stop"]

# 8. Policy Network Initialization (Multi-Model, LoRA, & Debiased Priors)
Loads the configured causal LM policy (`gpt2` in fp32 by default, or larger models like `Qwen/Qwen2.5-1.5B` in `bfloat16` + PEFT LoRA) and a fresh Adam optimizer.
When `debiased_logits = True`, computes frozen baseline logits $b_0$ at initialization over a neutral prefix (`"Prompt: Solve the task. Pipeline: ..."`) so Episode 0 begins near an unbiased $1/30$ uniform prior regardless of English letter-token frequencies.


In [ ]:
def trainable_parameters(model):
    params = [p for p in model.parameters() if p.requires_grad]
    return params if params else list(model.parameters())

def parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        if any(p.requires_grad for p in model.parameters()) and not parameter.requires_grad:
            continue
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().float().contiguous().numpy().tobytes())
    return digest.hexdigest()

def build_action_tokens(tokenizer, cfg):
    sampling = cfg.get("policy_sampling", "unique-signature-token-label-v1")
    if sampling == "restricted-first-token-multinomial":
        first_tokens = {action: tokenizer.encode(action, add_special_tokens=False)[0] for action in ACTIONS}
        cfg["signature_action_token_ids"] = {sig: first_tokens[sig] for sig in SIGNATURES}
        return first_tokens
    if sampling != "unique-signature-token-label-v1":
        raise ValueError(f"Unsupported policy_sampling: {sampling}")
    if len(SIGNATURES) > 26 or len(set(SIGNATURES)) != len(SIGNATURES):
        raise ValueError("Signature list must be unique and contain at most 26 entries")
    signature_tokens = {}
    for i, signature in enumerate(SIGNATURES):
        letter = chr(ord("A") + i)
        ids = tokenizer.encode(" " + letter, add_special_tokens=False)
        if len(ids) != 1:
            ids = tokenizer.encode(letter, add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(f"Signature label '{letter}' must encode to 1 token, got {ids}")
        signature_tokens[signature] = ids[0]
    if len(set(signature_tokens.values())) != len(signature_tokens):
        raise ValueError("Signature scoring labels must have distinct token IDs")
    first_tokens = {action: tokenizer.encode(action, add_special_tokens=False)[0] for action in ACTIONS}
    first_tokens.update(signature_tokens)
    cfg["signature_action_token_ids"] = dict(signature_tokens)
    return first_tokens

def compute_init_logit_bias(model, tokenizer, first_tokens, cfg):
    if not cfg.get("debiased_logits", False):
        return None
    device = next(model.parameters()).device
    max_pos = getattr(model.config, "n_positions", getattr(model.config, "max_position_embeddings", cfg["max_policy_tokens"]))
    bias = {}
    with torch.no_grad():
        for partial in ([], *[[m] for m in MODULES]):
            stage_key = " ".join(partial)
            choices = valid_actions(partial)
            text = f"Prompt: Solve the task. Pipeline: {stage_key}"
            encoded = tokenizer(text, return_tensors="pt", padding=True, truncation=True,
                                max_length=min(cfg["max_policy_tokens"], max_pos))
            encoded = {k: v.to(device) for k, v in encoded.items()}
            logits = model(**encoded, use_cache=False).logits[0, -1]
            ids = torch.tensor([first_tokens[a] for a in choices], device=device)
            bias[stage_key] = logits[ids].detach().cpu().float()
    return bias

def initialize_policy(cfg):
    if cfg["policy_device"] == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA policy requested but unavailable. Select the GPU kernel or explicitly use CPU")
    seed_everything(cfg["seed"])
    use_lora = bool(cfg.get("use_lora", False))
    try:
        if cfg["policy_model"] == "gpt2" and not use_lora:
            tokenizer = GPT2Tokenizer.from_pretrained(cfg["policy_model"], revision=cfg["policy_revision"])
            model = GPT2LMHeadModel.from_pretrained(cfg["policy_model"], revision=cfg["policy_revision"])
        else:
            tokenizer = AutoTokenizer.from_pretrained(cfg["policy_model"], revision=cfg["policy_revision"])
            dtype = torch.bfloat16 if (use_lora and cfg["policy_device"] == "cuda") else torch.float32
            model = AutoModelForCausalLM.from_pretrained(cfg["policy_model"], revision=cfg["policy_revision"], torch_dtype=dtype)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.truncation_side = "left"
    except Exception as exc:
        raise RuntimeError("Policy model files unavailable: " + str(exc)) from exc
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    if use_lora:
        from peft import LoraConfig, get_peft_model
        try:
            lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
                                  task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
            model = get_peft_model(model, lora_cfg)
        except ValueError:
            lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
                                  task_type="CAUSAL_LM", target_modules="all-linear")
            model = get_peft_model(model, lora_cfg)
    model.to(cfg["policy_device"]).eval()
    optimizer = torch.optim.Adam(trainable_parameters(model), lr=cfg["learning_rate"], foreach=False)
    first_tokens = build_action_tokens(tokenizer, cfg)
    init_logit_bias = compute_init_logit_bias(model, tokenizer, first_tokens, cfg)
    initial_digest = parameter_digest(model)
    return model, tokenizer, optimizer, first_tokens, initial_digest, init_logit_bias

def release_experiment(session):
    if session is not None:
        for key in ("policy", "optimizer", "tokenizer", "init_logit_bias"):
            session.pop(key, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# 9. Policy Action Generation, Without-Replacement Rescue & Analytical Pre-Allocation
Computes exact hierarchical categorical probabilities over the 30 `(Module, Signature)` pairs via debiased logits $z_\theta - b_0$.
- **Pillar 1 (Analytical Pre-Allocation — Layer B):** Evaluates expected unique coverage $\mathbb{E}[U(K)] = \sum_{j=1}^{30}[1 - (1 - \pi_j)^K]$ over `cfg["adaptive_levels"]` (`{2, 4, 8, 12}`) and solves $\max_K [\mathbb{E}[U(K)] - \lambda K]$ ($\lambda = 0.25$).
- **Pillar 3b (Without-Replacement Diversity Rescue):** When Stage 0 ($K=2$) fails and expands to $4/8/12$, `sample_untried_pipeline()` samples strictly untried `(Module, Signature)` pairs proportional to $\pi_\theta / Z_{\mathcal{U}}$, recording exact unmasked `log_probs` for `replay_log_probs` integrity and bounded importance weight $w_i = Z_{\mathcal{U}} \in (0, 1]$.


In [ ]:
def action_log_probs(model, tokenizer, first_tokens, prompt, partial, cfg, init_logit_bias=None):
    choices = valid_actions(partial)
    text = f"Prompt: {prompt} Pipeline: {' '.join(partial)}"
    max_pos = getattr(model.config, "n_positions", getattr(model.config, "max_position_embeddings", cfg["max_policy_tokens"]))
    encoded = tokenizer(text, return_tensors="pt", padding=True, truncation=True,
                        max_length=min(cfg["max_policy_tokens"], max_pos))
    device = next(model.parameters()).device
    encoded = {key: value.to(device) for key, value in encoded.items()}
    logits = model(**encoded, use_cache=False).logits[0, -1]
    ids = torch.tensor([first_tokens[action] for action in choices], device=device)
    selected = logits[ids].float()
    if init_logit_bias is not None:
        stage_key = " ".join(partial)
        if stage_key in init_logit_bias:
            selected = selected - init_logit_bias[stage_key].to(device=device, dtype=selected.dtype)
    return choices, torch.log_softmax(selected, dim=0)

def sample_actions(model, tokenizer, first_tokens, prompt, cfg, greedy=False, init_logit_bias=None):
    partial, log_probs = [], []
    with torch.no_grad():
        for _ in range(cfg["max_pipeline_steps"]):
            choices, scores = action_log_probs(model, tokenizer, first_tokens, prompt, partial, cfg, init_logit_bias=init_logit_bias)
            if greedy:
                index = int(torch.argmax(scores).item())
            else:
                index = int(torch.multinomial(scores.exp(), 1).item())
            action = choices[index]
            partial.append(action); log_probs.append(float(scores[index]))
            if action == "stop":
                break
    return dict(actions=partial, log_probs=log_probs)

def replay_log_probs(model, tokenizer, first_tokens, prompt, trajectory, cfg, init_logit_bias=None):
    partial, log_probs = [], []
    for action in trajectory["actions"]:
        choices, scores = action_log_probs(model, tokenizer, first_tokens, prompt, partial, cfg, init_logit_bias=init_logit_bias)
        if action not in choices:
            raise ValueError("Trajectory action violates released action grammar")
        log_probs.append(scores[choices.index(action)])
        partial.append(action)
    return log_probs

def action_collision_report(first_tokens):
    groups = {}
    for signature in SIGNATURES:
        groups.setdefault(first_tokens[signature], []).append(signature)
    return {token: actions for token, actions in groups.items() if len(actions) > 1}

def compute_pipeline_distribution(model, tokenizer, first_tokens, prompt, cfg, init_logit_bias=None):
    """Computes exact categorical probabilities pi_j over all 30 (Module x Signature) pipelines."""
    with torch.no_grad():
        _, mod_scores = action_log_probs(model, tokenizer, first_tokens, prompt, [], cfg, init_logit_bias=init_logit_bias)
        mod_probs = mod_scores.exp().cpu().numpy()
        pi = {}
        for m_idx, m in enumerate(MODULES):
            _, sig_scores = action_log_probs(model, tokenizer, first_tokens, prompt, [m], cfg, init_logit_bias=init_logit_bias)
            sig_probs = sig_scores.exp().cpu().numpy()
            for s_idx, s in enumerate(SIGNATURES):
                pi[(m, s)] = float(mod_probs[m_idx] * sig_probs[s_idx])
        prob_array = np.array(list(pi.values()), dtype=np.float64)
        prob_array = prob_array / prob_array.sum()
        return pi, prob_array

def exact_policy_entropy(model, tokenizer, first_tokens, prompt, cfg, init_logit_bias=None):
    """Differentiable exact Shannon entropy H(M, S | x) = H(M | x) + sum_m P(m|x) H(S | x, m) in nats."""
    _, mod_logp = action_log_probs(model, tokenizer, first_tokens, prompt, [], cfg, init_logit_bias=init_logit_bias)
    mod_p = mod_logp.exp()
    h_total = -(mod_p * mod_logp).sum()
    for m_idx, m in enumerate(MODULES):
        _, sig_logp = action_log_probs(model, tokenizer, first_tokens, prompt, [m], cfg, init_logit_bias=init_logit_bias)
        sig_p = sig_logp.exp()
        h_sig = -(sig_p * sig_logp).sum()
        h_total = h_total + mod_p[m_idx] * h_sig
    return h_total

def sample_untried_pipeline(model, tokenizer, first_tokens, prompt, tried_set, cfg, init_logit_bias=None):
    """Sample strictly untried (Module, Signature) from pi_theta / Z_U with exact unmasked log_probs and w_i = Z_U."""
    with torch.no_grad():
        _, mod_scores = action_log_probs(model, tokenizer, first_tokens, prompt, [], cfg, init_logit_bias=init_logit_bias)
        mod_logp = {m: float(mod_scores[i]) for i, m in enumerate(MODULES)}
        sig_logp = {}
        pi = {}
        for m in MODULES:
            _, sig_scores = action_log_probs(model, tokenizer, first_tokens, prompt, [m], cfg, init_logit_bias=init_logit_bias)
            for j, s in enumerate(SIGNATURES):
                lp_s = float(sig_scores[j])
                sig_logp[(m, s)] = lp_s
                pi[(m, s)] = math.exp(mod_logp[m] + lp_s)
    untried = [(m, s) for (m, s) in pi if (m, s) not in tried_set]
    if not untried:
        traj = generate_pipeline(model, tokenizer, first_tokens, prompt, cfg, greedy=False, init_logit_bias=init_logit_bias)
        traj["is_rescue_sample"] = False
        traj["importance_weight"] = 1.0
        return traj
    z_u = float(sum(pi[pair] for pair in untried))
    if z_u > 1e-12:
        weights = torch.tensor([pi[pair] / z_u for pair in untried], dtype=torch.float64)
        idx = int(torch.multinomial(weights, 1).item())
        w_i = float(min(1.0, max(1e-12, z_u)))
    else:
        weights = torch.ones(len(untried), dtype=torch.float64) / len(untried)
        idx = int(torch.multinomial(weights, 1).item())
        w_i = 1e-6
    m_star, s_star = untried[idx]
    return dict(
        actions=[m_star, s_star, "stop"],
        log_probs=[mod_logp[m_star], sig_logp[(m_star, s_star)], 0.0],
        module=m_star,
        signature=s_star,
        is_rescue_sample=True,
        importance_weight=w_i,
    )

def solve_preallocation_ceiling(prob_array, cfg, historical_pass_rate=None):
    """Pillar 1: Solve max_K [ E[U(K)] - lambda * K ] over adaptive_levels."""
    levels = cfg.get("adaptive_levels", [2, 4, 8, 12])
    lambda_penalty = cfg.get("lambda_penalty", 0.25)
    expected_coverage = {}
    utility = {}
    for k in levels:
        eu_k = float(np.sum(1.0 - (1.0 - prob_array) ** k))
        expected_coverage[k] = eu_k
        utility[k] = eu_k - lambda_penalty * k
    entropy = -float(np.sum(prob_array * np.log(prob_array + 1e-12)))
    if historical_pass_rate is not None:
        p_hat, attempts = historical_pass_rate
        if attempts >= 2 and p_hat == 0.0:
            return levels[0], expected_coverage, utility, entropy
    chosen_k = levels[0]
    for i in range(len(levels) - 1):
        k_curr, k_next = levels[i], levels[i + 1]
        if expected_coverage[k_next] - expected_coverage[k_curr] >= lambda_penalty * (k_next - k_curr):
            chosen_k = k_next
        else:
            break
    return chosen_k, expected_coverage, utility, entropy


# 10. DSPy Pipeline Generation
`generate_pipeline()` always receives the intended policy explicitly. This corrects the original evaluator's accidental use of a global policy. Generated actions and signatures are logged for every attempted candidate.

In [ ]:
def generate_pipeline(model, tokenizer, first_tokens, prompt, cfg, greedy=False, init_logit_bias=None):
    trajectory = sample_actions(model, tokenizer, first_tokens, prompt, cfg, greedy=greedy, init_logit_bias=init_logit_bias)
    actions = trajectory["actions"]
    trajectory["module"] = actions[0] if actions else None
    trajectory["signature"] = actions[1] if len(actions) > 1 else None
    trajectory["is_rescue_sample"] = False
    trajectory["importance_weight"] = 1.0
    return trajectory


# 11. DSPy Pipeline Execution & Deduplication Caching
Candidates execute sequentially. The original square-bracket parser and standard DSPy LM invocation are unchanged.

**Pillar 2 (Layer A — Intra-Episode Deduplication & Deterministic Caching):**
AutoDSPy candidate execution at `temperature = 0.0` is deterministic. Inside each episode, an intra-episode cache `(module, signature) -> (result, reward)` eliminates duplicate calls to Ollama. When duplicate pipelines are drawn within a group, Llama-3.1 inference runs only once per unique pipeline; duplicate candidates reuse cached results without incurring redundant GPU compute, while preserving exact group statistics ($K$) for GRPO advantage computation.


In [ ]:
def execute_pipeline(trajectory, prompt, lm):
    started = time.perf_counter()
    try:
        module, signature = trajectory["module"],trajectory["signature"]
        if module not in MODULES or signature not in SIGNATURES:
            raise ValueError("Invalid pipeline module or signature")
        input_field, output_field = [part.strip() for part in signature.split("->")]
        program = dspy.ChainOfThought(signature) if module == "CoT" else dspy.Predict(signature)
        instruction = "system instruction: Must give your final answer in square brackets without fail! e.g., [final answer] like this: [5] \n prompt: "
        with dspy.context(lm=lm):
            response = program(**{input_field:instruction+prompt})
        output = response.get(output_field)
        if output is None:
            raise ValueError("DSPy returned no signature output")
        return dict(output=str(output),success=True,error=None,seconds=time.perf_counter()-started)
    except Exception as exc:
        return dict(output=None,success=False,error=f"{type(exc).__name__}: {exc}",seconds=time.perf_counter()-started)

# 12. Reward Calculation
HotpotQA answer exact match normalizes casing, punctuation, articles, and whitespace. In `original_semantic` (default), when exact match fails, a semantic judge LLM call evaluates answer equivalence in $[0.0, 1.0]$. `answer_exact` disables the semantic fallback and scores only normalized exact string matches as $1.0$ or $0.0$.

In [ ]:
import re
import string
import collections

def normalize_answer(text):
    """Official HotpotQA answer normalization: case, ASCII punctuation, articles, whitespace."""
    value = str(text).lower()
    value = ''.join(character for character in value if character not in string.punctuation)
    value = re.sub(r"\b(a|an|the)\b", " ", value)
    return " ".join(value.split())

def extract_answer(response):
    if response is None:
        return None
    text = str(response).strip()
    brackets = re.findall(r"\[([^\[\]]+)\]", text)
    return brackets[-1].strip() if brackets else text

def compare_answers(response, target):
    prediction = extract_answer(response)
    return prediction is not None and normalize_answer(prediction) == normalize_answer(target)

def answer_f1(prediction, target):
    """Official evaluator's answer-only F1, precision and recall; not a training reward."""
    if prediction is None:
        return 0.0, 0.0, 0.0
    prediction, target = normalize_answer(prediction), normalize_answer(target)
    pred_tokens, target_tokens = prediction.split(), target.split()
    common = collections.Counter(pred_tokens) & collections.Counter(target_tokens)
    num_same = sum(common.values())
    if len(pred_tokens) == 0 or len(target_tokens) == 0:
        same = float(prediction == target)
        return same, same, same
    if num_same == 0:
        return 0.0, 0.0, 0.0
    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(target_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1, precision, recall

def extract_text_answer(response):
    """Explicit final text, or a plain single-line answer; no arbitrary reasoning suffix."""
    if response is None:
        return None
    text = str(response).strip()
    if not text:
        return None
    marked = list(re.finditer(r'(?:^|\n)\s*(?:the\s+)?(?:final\s+answer|answer|final\s+result)\s*(?:is\b|:|=)\s*', text, re.I))
    if '####' in text:
        text = text.rsplit('####', 1)[1].strip()
    elif marked:
        text = text[marked[-1].end():].strip()
    elif re.search(r'\[[^\[\]]+\]\s*[.!]?\s*$', text):
        match = re.search(r'\[([^\[\]]+)\]\s*[.!]?\s*$', text)
        prefix = text[:match.start()]
        if '[' in prefix or re.search(r'\b(?:intermediate|partial|previous)\b', prefix, re.I):
            return None
        text = match.group(1).strip()
    elif '\n' in text:
        return None
    text = text.strip()
    for _ in range(3):
        before = text
        for left, right in [('**','**'), ('`','`'), ('[',']')]:
            if text.startswith(left) and text.endswith(right) and len(text) > len(left)+len(right):
                text = text[len(left):-len(right)].strip()
        if text == before:
            break
    return text if text and '\n' not in text and normalize_answer(text) else None

def reward_prediction(response, cfg=None):
    mode = (cfg or {}).get("reward_mode", "original_semantic")
    return extract_text_answer(response) if mode == "answer_exact" else extract_answer(response)

def parse_judge_score(text):
    value = str(text).strip()
    bracket = re.fullmatch(r"\s*\[\s*([^\[\]]+)\s*\]\s*", value)
    if bracket:
        value = bracket.group(1).strip()
    if re.fullmatch(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)", value) is None:
        raise ValueError("Judge did not return a single finite numeric score")
    score = float(value)
    if not math.isfinite(score):
        raise ValueError("Nonfinite judge score")
    return min(1.0, max(0.0, score))

def compute_reward(prompt, response, target, lm, cfg=None, reward_context=None, identifiers=None):
    mode = (cfg or {}).get("reward_mode", "original_semantic")
    if mode not in ("original_semantic", "answer_exact"):
        raise ValueError("Unknown reward mode")
    prediction = reward_prediction(response, cfg)
    norm_pred = normalize_answer(prediction) if prediction is not None else ""
    norm_tgt = normalize_answer(target)
    exact = prediction is not None and norm_pred == norm_tgt
    target_in_pred = bool(prediction is not None and norm_tgt and norm_tgt in norm_pred)
    f1, prec, rec = answer_f1(prediction, target)
    if response is None:
        return dict(reward=0.0, exact_match=False, answer_f1=0.0, target_in_prediction=False,
                    semantic_used=False, judge_score=None, judge_error=None)
    if exact:
        return dict(reward=1.0, exact_match=True, answer_f1=1.0, target_in_prediction=True,
                    semantic_used=False, judge_score=None, judge_error=None)
    if mode == "answer_exact":
        return dict(reward=0.0, exact_match=False, answer_f1=f1, target_in_prediction=target_in_pred,
                    semantic_used=False, judge_score=None, judge_error=None)
    judge_prompt = (
        "Evaluate whether the following response correctly answers the prompt based on the ground truth and give your final score in the square brackets[final score]. only the value in the [] should in your response and nothing else.\n"
        f"Prompt: {prompt}\nResponse: {response}\nGround Truth: {target}\n"
        "Return a score in range between 1.0 to 0.0 if the response is correct or partially correct (matches or is equivalent to the ground truth), or 0.0 if incorrect.\nfinal score:[]"
    )
    try:
        outputs = lm(judge_prompt)
        score = parse_judge_score(outputs[0])
        return dict(reward=score, exact_match=False, answer_f1=f1, target_in_prediction=target_in_pred,
                    semantic_used=True, judge_score=score, judge_error=None)
    except Exception as exc:
        return dict(reward=0.0, exact_match=False, answer_f1=f1, target_in_prediction=target_in_pred,
                    semantic_used=True, judge_score=None, judge_error=f"{type(exc).__name__}: {exc}")


# 13. Figure 2 GRPO Loss ($1/K$ Normalized), Entropy Floor, & Unified Contrast Masking
$$\bar r = \frac{1}{K}\sum_{i=1}^K r_i,\quad A_i = r_i - \bar r,\quad \mathcal L_{\text{RL}} = -\frac{1}{K}\sum_{i=1}^K w_i \sum_t\log\pi_\theta(a_{i,t}\mid s_{i,t})A_i - \lambda_H \mathcal{H}(\pi_\theta(\cdot\mid x))$$

**APEX-GRPO v3 Optimizer Controls:**
1. **Bounded-IS $1/K$ Loss Normalization:** Scales by $1/K$ (unless disabled in `static_k5_original`) and weights rescue samples by $w_i = Z_{\mathcal{U}} \in (0, 1]$ (`w_i = 1.0` on-policy in Stage 0).
2. **Unified Contrast Gating (Layer D):** When `use_zero_var_mask = True`, skips `optimizer.step()` if all advantages are zero (`zero_variance_group`) OR if `max(r) - min(r) <= reward_contrast` (`below_contrast_threshold`), blocking fractional judge-jitter drift.
3. **Entropy Floor & Gradient Clipping:** Mild exact 30-action entropy regularization ($\lambda_H = 0.02$) prevents single-arm collapse while `clip_grad_norm_(..., max_norm=1.0)` prevents early gradient spikes.


In [ ]:
def group_advantages(rewards):
    values = np.asarray(rewards, dtype=np.float64)
    if len(values) == 0 or not np.isfinite(values).all():
        raise ValueError("GRPO requires a nonempty finite group")
    return float(values.mean()), values - values.mean()

def figure2_loss(log_prob_trajectories, advantages, group_size=None, normalize_by_k=True, weights=None):
    if len(log_prob_trajectories) != len(advantages):
        raise ValueError("One advantage per candidate is required")
    K = float(group_size if group_size is not None else len(log_prob_trajectories))
    if K <= 0:
        raise ValueError("Group size must be positive")
    if weights is None:
        weights = [1.0] * len(log_prob_trajectories)
    raw = -sum(float(w) * sum(log_probs) * float(advantage)
               for log_probs, advantage, w in zip(log_prob_trajectories, advantages, weights))
    return (1.0 / K) * raw if normalize_by_k else raw

def update_policy(session, prompt, trajectories, rewards):
    if session.get("evaluation_only", False):
        raise RuntimeError("Evaluation-only checkpoint: use explicit resume before updating")
    model, optimizer = session["policy"], session["optimizer"]
    cfg = session["config"]
    K = len(trajectories)
    if K not in cfg["adaptive_levels"] or len(rewards) != K:
        raise ValueError(f"Episode must contain valid group size from {cfg['adaptive_levels']}, got {K}")
    values = np.asarray(rewards, dtype=np.float64)
    mean, advantages = group_advantages(rewards)
    spread = float(values.max() - values.min())
    is_zero_variance = bool(np.all(advantages == 0))

    if cfg.get("use_zero_var_mask", True):
        if is_zero_variance:
            optimizer.zero_grad(set_to_none=True)
            return dict(
                group_mean=mean, advantages=advantages.tolist(), loss=0.0,
                gradient_norm=0.0, raw_gradient_norm=0.0, policy_entropy_exact=None,
                zero_advantage_group=True, optimizer_step=False, skip_reason="zero_variance_group",
            )
        if spread <= cfg.get("reward_contrast", 0.0):
            optimizer.zero_grad(set_to_none=True)
            return dict(
                group_mean=mean, advantages=advantages.tolist(), loss=0.0,
                gradient_norm=0.0, raw_gradient_norm=0.0, policy_entropy_exact=None,
                zero_advantage_group=False, optimizer_step=False, skip_reason="below_contrast_threshold",
            )

    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    model.eval()
    params = trainable_parameters(model)
    param_dtype = next(iter(params)).dtype if params else torch.float32
    replay_tol = 5e-3 if param_dtype == torch.bfloat16 else 2e-5
    init_bias = session.get("init_logit_bias")
    norm_by_k = cfg.get("normalize_loss_by_k", True)
    use_is = cfg.get("rescue_is_correction", True)

    for trajectory, advantage in zip(trajectories, advantages):
        scores = replay_log_probs(model, session["tokenizer"], session["first_tokens"], prompt, trajectory, cfg, init_logit_bias=init_bias)
        replay_values = [float(score.detach()) for score in scores]
        if not np.allclose(replay_values, trajectory["log_probs"], rtol=replay_tol, atol=replay_tol):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError("Replay differs from rollout; parameters/mode/state changed before update")
        w_i = float(trajectory.get("importance_weight", 1.0)) if use_is else 1.0
        loss = figure2_loss([scores], [advantage], group_size=K, normalize_by_k=norm_by_k, weights=[w_i])
        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError("Nonfinite Figure 2 loss; no optimizer step permitted")
        total_loss += float(loss.detach())
        loss.backward()
        del loss, scores

    entropy_val = None
    entropy_coef = float(cfg.get("entropy_coef", 0.0) or 0.0)
    if entropy_coef > 0.0:
        ent = exact_policy_entropy(model, session["tokenizer"], session["first_tokens"], prompt, cfg, init_logit_bias=init_bias)
        entropy_val = float(ent.detach())
        ent_loss = -entropy_coef * ent
        if torch.isfinite(ent_loss):
            total_loss += float(ent_loss.detach())
            ent_loss.backward()
        del ent, ent_loss

    raw_norm_sq = sum(float(p.grad.detach().double().square().sum()) for p in params if p.grad is not None)
    raw_norm = math.sqrt(raw_norm_sq)
    if not math.isfinite(raw_norm):
        optimizer.zero_grad(set_to_none=True)
        raise RuntimeError("Nonfinite gradient; no optimizer step permitted")

    clip_norm = cfg.get("grad_clip_norm")
    if clip_norm is not None:
        torch.nn.utils.clip_grad_norm_(params, float(clip_norm))
        norm_sq = sum(float(p.grad.detach().double().square().sum()) for p in params if p.grad is not None)
        norm = math.sqrt(norm_sq)
    else:
        norm = raw_norm

    optimizer.step()
    session.pop("loaded_checkpoint", None)
    optimizer.zero_grad(set_to_none=True)
    return dict(
        group_mean=mean, advantages=advantages.tolist(), loss=total_loss,
        gradient_norm=norm, raw_gradient_norm=raw_norm, policy_entropy_exact=entropy_val,
        zero_advantage_group=is_zero_variance, optimizer_step=True, skip_reason=None,
    )


# 14. GRPO Training Loop (APEX-GRPO v3 Pipeline)
Every episode uses one scheduled question and runs according to the selected `RUN_PROFILE`:
1. **Pre-Allocation (Pillar 1):** Computes exact 30-pipeline coverage and sets $K_{\text{ceiling}}$ (`2, 4, 8, 12`), unless `use_prealloc=False`.
2. **Intra-Episode Deduplication (Pillar 2):** Reuses deterministic `temp=0.0` results inside the episode when `use_dedup_cache=True`.
3. **Staged Exit & Without-Replacement Rescue (Pillar 3):** Starts at $K_{\min}=2$; exits on `contrast > reward_contrast`, `all_correct_exit`, or `non_failure_homogeneous`; expands via `sample_untried_pipeline()` when `use_without_replacement=True`.
4. **Contrast-Gated Masked Update (Pillar 4):** Updates with $1/K$ normalized IS-weighted loss, entropy floor, and gradient clipping.


In [ ]:
def initialize_experiment(cfg, create_run=True):
    cfg, datasets = load_hotpotqa(cfg)
    data = prepare_selection(cfg, datasets)
    if STATIC_REFERENCE is not None and data["selection_id"] != STATIC_REFERENCE["selection_id"]:
        raise ValueError("Dataset selection or episode schedule differs from the pinned Static-K run")
    lm, llm_metadata = initialize_ollama(cfg)
    if STATIC_REFERENCE is not None and llm_metadata.get("model", {}).get("digest") != STATIC_REFERENCE["execution_model"]["model"]["digest"]:
        raise ValueError("Ollama model digest differs from the pinned Static-K run")
    reward_runtime = None
    judge_lm = lm
    model, tokenizer, optimizer, first_tokens, digest, init_logit_bias = initialize_policy(cfg)
    if STATIC_REFERENCE is not None and digest != STATIC_REFERENCE["initial_policy_digest"]:
        raise ValueError("Initial policy parameters differ from the pinned Static-K run")
    eval_cache = DeterministicEvalCache(resolve_eval_cache_path(cfg)) if cfg.get("eval_cache_enabled", True) else None
    session = dict(
        config=cfg, data=data, lm=lm, llm_metadata=llm_metadata,
        reward_context=reward_runtime, judge_lm=judge_lm,
        policy=model, tokenizer=tokenizer, optimizer=optimizer,
        first_tokens=first_tokens, initial_policy_digest=digest,
        init_logit_bias=init_logit_bias, eval_cache=eval_cache,
        state=dict(episode=0, updates=0, train=[], validation=[], tests=[], best=None, seen=[],
                   cumulative_training_seconds=0.0),
        run_dir=None, in_episode=False,
    )
    if create_run:
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:8]
        root = Path(cfg["results_root"]) / cfg["dataset"] / cfg["run_profile"] / f"seed{cfg['seed']}" / cfg["preset"] / run_id
        root.mkdir(parents=True, exist_ok=False); (root / "checkpoints").mkdir()
        session["run_dir"] = root
        
        write_json(root / "config.json", dict(
            config=cfg, selection_id=data["selection_id"],
            selection_path=data["selection_path"], initial_policy_digest=digest,
            hardware=hardware_metadata(), execution_model=llm_metadata
        ))
    return session

def append_journal(session, event):
    if session["run_dir"] is None:
        return
    with (session["run_dir"] / "attempts.jsonl").open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(event, allow_nan=False) + "\n"); stream.flush(); os.fsync(stream.fileno())

def reserved_training_candidates(session):
    journal = session["run_dir"] / "attempts.jsonl"
    if not journal.exists():
        return 0
    return sum(row.get("kind") == "training_reservation" for row in read_jsonl(journal))

def train_grpo(session):
    if session.get("evaluation_only", False):
        raise RuntimeError("Evaluation-only session: initialize a new run or use explicit resume")
    cfg, state = session["config"], session["state"]
    validated = {row["episode"] for row in state["validation"]}
    if state["episode"] == cfg["num_episodes"] and (cfg["num_episodes"] in validated or state.get("budget_exhausted", False)):
        raise RuntimeError("Run completed; create a new run or evaluate its checkpoint")
    pending_validation = (state.get("budget_exhausted", False) or state["episode"] == 0 or state["episode"] % cfg["validation_interval"] == 0 or state["episode"] == cfg["num_episodes"]) and state["episode"] not in validated
    if pending_validation:
        validation = evaluate_module(session, "validation", episode=state["episode"])
        state["validation"].append(validation); save_checkpoint(session, validation=validation)
    elif not (session["run_dir"] / "latest.json").exists():
        save_checkpoint(session)
    if state["episode"] == cfg["num_episodes"] or state.get("budget_exhausted", False):
        write_summary(session)
        return session
    init_bias = session.get("init_logit_bias")
    try:
        while state["episode"] < cfg["num_episodes"]:
            remaining = None if cfg["candidate_budget"] is None else cfg["candidate_budget"] - reserved_training_candidates(session)
            if remaining is not None and remaining < cfg["adaptive_levels"][0]:
                state["budget_exhausted"] = True
                append_journal(session, dict(kind="budget_exhausted", remaining=remaining, episode=state["episode"]))
                validation = None
                if not any(row["episode"] == state["episode"] for row in state["validation"]):
                    validation = evaluate_module(session, "validation", episode=state["episode"])
                    state["validation"].append(validation)
                save_checkpoint(session, validation=validation); write_summary(session)
                break
            episode = state["episode"] + 1
            offset = session["data"]["selection"]["episode_pool_offsets"][episode - 1]
            example = session["data"]["subsets"]["train"][offset]
            prompt = example["input"]
            session["in_episode"] = True
            if next(session["policy"].parameters()).is_cuda:
                torch.cuda.reset_peak_memory_stats()
            begin = time.perf_counter(); call_start = len(session["lm"].events)

            # Pillar 1: Analytical Pre-Allocation (Layer B)
            pi_dict, prob_array = compute_pipeline_distribution(
                session["policy"], session["tokenizer"], session["first_tokens"], prompt, cfg, init_logit_bias=init_bias
            )
            q_history = session["state"].setdefault("question_history", {}).get(example["id"])
            levels = cfg["adaptive_levels"]
            if cfg.get("use_prealloc", True) and cfg.get("use_adaptive_k", True):
                k_ceiling, expected_coverage, utility_dict, policy_entropy = solve_preallocation_ceiling(
                    prob_array, cfg, historical_pass_rate=q_history
                )
            else:
                k_ceiling = levels[-1]
                expected_coverage = {k: float(np.sum(1.0 - (1.0 - prob_array) ** k)) for k in levels}
                policy_entropy = -float(np.sum(prob_array * np.log(prob_array + 1e-12)))

            # Pillar 2 (Dedup Cache) + Pillar 3 (Staged Waterfall & Without-Replacement Rescue)
            episode_cache = {}
            tried_pipelines = set()
            trajectories, results, reward_details = [], [], []
            stop_reason = None
            physical_calls_start = len(session["lm"].events)
            use_dedup = bool(cfg.get("use_dedup_cache", True))
            use_wor = bool(cfg.get("use_without_replacement", True))

            for stage_idx, target_k in enumerate(levels):
                needed = target_k - len(trajectories)
                for _ in range(needed):
                    candidate_idx = len(trajectories)
                    append_journal(session, dict(kind="training_reservation", episode=episode, candidate=candidate_idx))
                    candidate_call_start = len(session["lm"].events)

                    if stage_idx >= 1 and use_wor:
                        trajectory = sample_untried_pipeline(
                            session["policy"], session["tokenizer"], session["first_tokens"],
                            prompt, tried_pipelines, cfg, init_logit_bias=init_bias
                        )
                    else:
                        trajectory = generate_pipeline(
                            session["policy"], session["tokenizer"], session["first_tokens"],
                            prompt, cfg, greedy=False, init_logit_bias=init_bias
                        )
                    pipeline_key = (trajectory["module"], trajectory["signature"])
                    tried_pipelines.add(pipeline_key)

                    if use_dedup and pipeline_key in episode_cache:
                        cached_result, cached_reward = episode_cache[pipeline_key]
                        result = dict(cached_result); result["cached"] = True
                        reward = dict(cached_reward); reward["cached"] = True
                    else:
                        result = execute_pipeline(trajectory, prompt, session["lm"])
                        result["cached"] = False
                        reward = compute_reward(prompt, result["output"], example["answer"], session["lm"], cfg)
                        reward["cached"] = False
                        episode_cache[pipeline_key] = (result, reward)

                    trajectories.append(trajectory); results.append(result); reward_details.append(reward)
                    append_journal(session, dict(kind="training_attempt", episode=episode, candidate=candidate_idx,
                                                 question_id=example["id"], execution=result, reward=reward,
                                                 cached=result.get("cached", False),
                                                 resources=call_statistics(session["lm"].events[candidate_call_start:]),
                                                 timestamp=datetime.now(timezone.utc).isoformat()))
                    if cfg["candidate_budget"] is not None and reserved_training_candidates(session) >= cfg["candidate_budget"]:
                        stop_reason = "budget_exhausted"
                        break
                if stop_reason is not None:
                    break

                failures = any(not item["success"] for item in results) or any(item.get("judge_error") for item in reward_details)
                current_rewards = [item["reward"] for item in reward_details]
                decision = adaptive_decision(current_rewards, failures, cfg, k_ceiling=k_ceiling)
                if decision is not None:
                    stop_reason = decision
                    break

            rewards = [detail["reward"] for detail in reward_details]
            realized_k = len(rewards)

            mean_r = float(np.mean(rewards)) if rewards else 0.0
            prev_pass, prev_count = session["state"].setdefault("question_history", {}).get(example["id"], (0.0, 0))
            new_count = prev_count + 1
            new_pass = ((prev_pass * prev_count) + (1.0 if mean_r > cfg.get("failure_reward_threshold", 0.25) else 0.0)) / new_count
            session["state"]["question_history"][example["id"]] = (new_pass, new_count)

            # Pillar 4: Contrast-Gated Masked 1/K Update + Entropy Floor + Grad Clip
            update = update_policy(session, prompt, trajectories, rewards)

            seconds = time.perf_counter() - begin
            seen = set(state["seen"]); seen.add(example["id"])
            calls = call_statistics(session["lm"].events[call_start:])
            physical_lm_calls = len(session["lm"].events[physical_calls_start:])
            cached_calls = sum(r.get("cached", False) for r in results)
            unique_pipelines = len(set(tuple(t["actions"][:2]) for t in trajectories))
            rescue_samples = [t for t in trajectories if t.get("is_rescue_sample", False)]
            rescue_rewards = [w["reward"] for t, w in zip(trajectories, reward_details) if t.get("is_rescue_sample", False)]
            rescue_ems = [bool(c.get("exact_match", False)) for t, c in zip(trajectories, reward_details) if t.get("is_rescue_sample", False)]
            is_weights = [float(t.get("importance_weight", 1.0)) for t in rescue_samples]

            row = dict(
                dataset=cfg["dataset"],
                run_profile=cfg["run_profile"],
                k=realized_k,
                k_ceiling=k_ceiling,
                expected_coverage=expected_coverage,
                policy_entropy=policy_entropy,
                policy_entropy_exact=update.get("policy_entropy_exact"),
                unique_pipelines_count=unique_pipelines,
                unique_pipeline_ratio=float(unique_pipelines / realized_k) if realized_k > 0 else 0.0,
                physical_lm_calls=physical_lm_calls,
                cached_pipeline_executions=cached_calls,
                rescue_samples_count=len(rescue_samples),
                rescue_succeeded_contrast=bool(len(rescue_samples) > 0 and stop_reason == "contrast"),
                rescue_succeeded_em=int(sum(rescue_ems)),
                rescue_max_reward=float(max(rescue_rewards)) if rescue_rewards else None,
                mean_rescue_is_weight=float(np.mean(is_weights)) if is_weights else None,
                stop_reason=stop_reason,
                allocation_version=cfg["allocation_version"],
                seed=cfg["seed"],
                episode=episode,
                question_id=example["id"],
                task_type=example.get("task_type", cfg["dataset"]),
                reward_mode=effective_reward_config(cfg)["reward_mode"],
                reward_protocol_version=effective_reward_config(cfg)["reward_protocol_version"],
                question_pool_offset=offset,
                actions=[t["actions"] for t in trajectories],
                signatures=[t["signature"] for t in trajectories],
                rewards=rewards,
                mean_group_reward=update["group_mean"],
                reward_std=float(np.std(rewards)),
                reward_variance=float(np.var(rewards)),
                reward_spread=float(max(rewards) - min(rewards)) if rewards else 0.0,
                advantages=update["advantages"],
                zero_advantage_group=update["zero_advantage_group"],
                optimizer_step=update["optimizer_step"],
                skip_reason=update.get("skip_reason"),
                grpo_loss=update["loss"],
                gradient_norm=update["gradient_norm"],
                raw_gradient_norm=update.get("raw_gradient_norm", update["gradient_norm"]),
                optimizer_update_count=state["updates"] + int(update["optimizer_step"]),
                unique_training_questions=len(seen),
                attempted_pipeline_evaluations=realized_k,
                successful_pipeline_evaluations=sum(result["success"] for result in results),
                failed_pipeline_evaluations=sum(not result["success"] for result in results),
                episode_seconds=seconds,
                cumulative_training_seconds=state["cumulative_training_seconds"] + seconds,
                average_pipeline_execution_latency=float(np.mean([result["seconds"] for result in results])),
                pipeline_execution_seconds=sum(result["seconds"] for result in results),
                candidate_details=[dict(trajectory=t, execution=r, reward=w) for t, r, w in zip(trajectories, results, reward_details)],
                out_of_memory_failures=sum("out of memory" in (result["error"] or "").lower() for result in results),
                execution_failures=[result["error"] for result in results if result["error"]],
                judge_failures=[detail["judge_error"] for detail in reward_details if detail["judge_error"]],
                ollama_memory=ollama_snapshot(cfg),
                **calls,
                **gpu_statistics(session["policy"])
            )
            state["train"].append(row)
            state["episode"] = episode
            if update["optimizer_step"]:
                state["updates"] += 1
            state["seen"] = sorted(seen)
            state["cumulative_training_seconds"] += seconds
            session["in_episode"] = False
            synchronize_logs(session)

            validation = None
            if episode % cfg["validation_interval"] == 0 or episode == cfg["num_episodes"]:
                validation = evaluate_module(session, "validation", episode=episode)
                state["validation"].append(validation)
                print(f"--> [Validation Ep {episode}] EM: {validation['exact_match_accuracy']*100:.1f}% | F1: {validation['answer_f1']:.3f} | Substring: {validation['target_in_prediction_rate']*100:.1f}% | Mean Reward (Semantic): {validation['mean_reward']:.3f} ({validation['mean_reward']*100:.1f}%)")
            if validation is not None or episode % cfg["checkpoint_interval"] == 0 or episode == cfg["num_episodes"]:
                save_checkpoint(session, validation=validation)
            write_summary(session)
            print(f"K={realized_k} (ceil={k_ceiling}, uniq={unique_pipelines}, rescue={len(rescue_samples)}) ep={episode}/{cfg['num_episodes']} "
                  f"rew={row['mean_group_reward']:.3f} loss={row['grpo_loss']:.4f} grad={row['gradient_norm']:.3f} "
                  f"step={row['optimizer_step']} ({row['skip_reason'] or row['stop_reason']})")
    except BaseException as exc:
        append_journal(session, dict(kind="interruption", episode=state["episode"] + int(session["in_episode"]),
                       error=f"{type(exc).__name__}: {exc}", in_episode=session["in_episode"],
                       out_of_memory="out of memory" in str(exc).lower(),
                       gpu_memory=gpu_statistics(session["policy"]),
                       timestamp=datetime.now(timezone.utc).isoformat()))
        if not session["in_episode"]:
            save_checkpoint(session)
            write_summary(session)
        else:
            print("Episode interrupted. Resume the latest.json checkpoint; incomplete work was not committed.")
        raise
    return session


# 15. Checkpoint Management
Immutable filenames prevent accidental checkpoint overwrite; atomic latest/best pointers stay in the explicit run. Select best by highest validation numeric EM; **earliest episode wins ties**. Every validation state, including episode 0, is checkpointed. Resume only the latest durable checkpoint; historical checkpoints are available for evaluation.

Checkpoints contain policy, Adam, complete config, all RNG states, selection/schedule, metrics and execution identity. Load only your own trusted checkpoints (`weights_only=False` permits Python/NumPy RNG restoration). Final-test results never select checkpoints.

In [ ]:
def export_policy_state(session):
    model = session["policy"]
    if session["config"].get("use_lora", False):
        trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
        return {k: v.detach().cpu() for k, v in model.state_dict().items() if any(tn in k or k == tn for tn in trainable_names)}
    return model.state_dict()

def checkpoint_payload(session):
    if session.get("evaluation_only", False):
        raise RuntimeError("Do not save training checkpoints from an evaluation-only session")
    if session["in_episode"]:
        raise RuntimeError("Cannot checkpoint an incomplete episode")
    init_bias = session.get("init_logit_bias")
    serialized_bias = {k: v.detach().cpu() for k, v in init_bias.items()} if init_bias is not None else None
    return dict(
        format_version=2,
        config=copy.deepcopy(session["config"]),
        policy=export_policy_state(session),
        policy_trainable_only=bool(session["config"].get("use_lora", False)),
        init_logit_bias=serialized_bias,
        optimizer=session["optimizer"].state_dict(),
        state=copy.deepcopy(session["state"]),
        rng=rng_state(),
        selection=copy.deepcopy(session["data"]["selection"]),
        selection_id=session["data"]["selection_id"],
        initial_policy_digest=session["initial_policy_digest"],
        execution_model=session["llm_metadata"],
        hardware=hardware_metadata(),
        run_dir=str(session["run_dir"]),
        saved_at=datetime.now(timezone.utc).isoformat(),
        
    )

def atomic_torch_save(path, payload):
    path = Path(path)
    if path.exists():
        raise FileExistsError(f"Refusing checkpoint overwrite: {path}")
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    with temporary.open("wb") as stream:
        torch.save(payload, stream); stream.flush(); os.fsync(stream.fileno())
    os.link(temporary, path); temporary.unlink()

def save_checkpoint(session, validation=None):
    state = session["state"]; root = session["run_dir"]
    if root is None:
        raise RuntimeError("No active run directory")
    path = root / "checkpoints" / f"episode_{state['episode']:04d}_{uuid.uuid4().hex[:8]}.pt"
    old_best = state["best"]
    improved = validation is not None and (old_best is None or validation["exact_match_accuracy"] > old_best["score"])
    if improved:
        state["best"] = dict(
            score=validation["exact_match_accuracy"],
            exact_match_accuracy=validation["exact_match_accuracy"],
            answer_f1=validation.get("answer_f1"), mean_reward=validation.get("mean_reward"),
            episode=state["episode"],
            checkpoint=str(path),
        )
    try:
        atomic_torch_save(path, checkpoint_payload(session))
    except BaseException:
        state["best"] = old_best
        raise
    write_json(root / "latest.json", dict(checkpoint=str(path), episode=state["episode"]), overwrite=True)
    if improved:
        write_json(root / "best.json", state["best"], overwrite=True)
    synchronize_logs(session)
    backup_results(session["config"])
    return path

def compatible_config(expected, actual):
    ignored = {"results_root", "eval_cache_enabled"}
    left = canonical_experiment_config(expected); right = canonical_experiment_config(actual)
    differing = [key for key in set(left["config"]) | set(right["config"])
                 if key not in ignored and left["config"].get(key) != right["config"].get(key)]
    if left["reward"] != right["reward"]:
        differing.append("reward protocol/mode/cache/judge identity")
    if differing:
        raise ValueError("Checkpoint configuration mismatch: " + ", ".join(sorted(differing)))

def load_checkpoint_into(session, path, for_resume=False):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Selected checkpoint not found: {path}")
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if payload.get("format_version") not in (1, 2):
        raise ValueError("Unsupported checkpoint format")
    compatible_config(session["config"], payload["config"])
    if payload["selection_id"] != session["data"]["selection_id"] or payload["selection"] != session["data"]["selection"]:
        raise ValueError("Checkpoint dataset/schedule mismatch")
    if payload["initial_policy_digest"] != session["initial_policy_digest"]:
        raise ValueError("Initial policy parameters differ")
    old_model = payload["execution_model"].get("model", {})
    new_model = session["llm_metadata"].get("model", {})
    if old_model.get("digest") != new_model.get("digest"):
        raise ValueError("Ollama model digest changed; refuse controlled resume/evaluation")
    if payload["hardware"]["packages"] != VERSIONS or payload["execution_model"].get("version") != session["llm_metadata"].get("version"):
        raise ValueError("Dependency/Ollama versions changed; refuse controlled checkpoint loading")
    root = Path(payload["run_dir"])
    if for_resume:
        if not root.exists():
            raise FileNotFoundError("Original run directory unavailable")
        latest = json.loads((root / "latest.json").read_text(encoding="utf-8"))
        if Path(latest["checkpoint"]).resolve() != path.resolve():
            raise ValueError("Resume only latest durable checkpoint")
        if payload["state"]["episode"] >= payload["config"]["num_episodes"] and any(row["episode"] == payload["config"]["num_episodes"] for row in payload["state"]["validation"]):
            raise ValueError("Run completed; cannot resume as a longer/different preset")
    
    if session["config"].get("debiased_logits", False) and "init_logit_bias" in payload:
        session["init_logit_bias"] = payload["init_logit_bias"]
    strict_load = not bool(payload.get("policy_trainable_only", False))
    session["policy"].load_state_dict(payload["policy"], strict=strict_load); session["policy"].eval()
    if for_resume:
        session["optimizer"].load_state_dict(payload["optimizer"])
    session["state"] = payload["state"]; session["run_dir"] = root; session["in_episode"] = False
    session["loaded_checkpoint"] = str(path)
    session["evaluation_only"] = not for_resume
    restore_rng(payload["rng"])
    if for_resume:
        synchronize_logs(session)
        if session["state"]["best"] is not None:
            write_json(root / "best.json", session["state"]["best"], overwrite=True)
        write_summary(session)
        append_journal(session, dict(kind="resume", next_episode=session["state"]["episode"] + 1,
                                   checkpoint=str(path), timestamp=datetime.now(timezone.utc).isoformat()))
    del payload; gc.collect()
    return session

def load_selected_experiment(path, cfg, for_resume=False):
    session = initialize_experiment(cfg, create_run=False)
    return load_checkpoint_into(session, path, for_resume=for_resume)


# 16. Validation
One sampled pipeline per fixed validation question, no gradients or optimizer updates. A fixed split-specific policy RNG seed is reused at each checkpoint, then the training RNG is restored. Llama uses fixed seed/temperature; version/device changes can still affect outputs. Strict numeric EM, fractional semantic scores and failures remain distinct.

In [ ]:
def evaluate_module(session, split="validation", episode=None, examples=None, policy_override_fn=None):
    if split not in ("validation", "test", "full_test"):
        raise ValueError("Evaluation must use validation/test, never training")
    if examples is None:
        if split == "full_test":
            raise ValueError("Full test requires explicit examples")
        examples = session["data"]["subsets"][split]
    if not examples:
        raise ValueError("Empty evaluation set")
    train_ids = set(session["data"]["selection"]["ids"]["train"])
    if any(example["id"] in train_ids for example in examples):
        raise ValueError("Training examples cannot enter held-out evaluation")
    cfg = session["config"]; model = session["policy"]
    old_mode = model.training; model.eval()
    call_start = len(session["lm"].events); begin = time.perf_counter(); records = []
    seed = cfg["seed"] + cfg["evaluation_seed_offset"] + (0 if split == "validation" else 1)
    greedy = (cfg.get("eval_decoding", "greedy") == "greedy")
    init_bias = session.get("init_logit_bias")
    eval_cache = session.get("eval_cache") if cfg.get("eval_cache_enabled", True) else None
    cache_hits = 0
    try:
        with isolated_rng(seed), torch.no_grad():
            for example in examples:
                question_call_start = len(session["lm"].events)
                prompt = example["input"]
                if policy_override_fn is not None:
                    trajectory = policy_override_fn(example)
                else:
                    trajectory = generate_pipeline(model, session["tokenizer"], session["first_tokens"],
                                                   prompt, cfg, greedy=greedy, init_logit_bias=init_bias)
                cache_key = None
                cached_pair = None
                if eval_cache is not None and cfg["execution_temperature"] == 0.0:
                    cache_key = eval_cache.make_key(cfg, session.get("llm_metadata"), example["id"],
                                                    trajectory["module"], trajectory["signature"])
                    cached_pair = eval_cache.get(cache_key)
                if cached_pair is not None:
                    execution, reward = cached_pair
                    execution["cached"] = True
                    cache_hits += 1
                else:
                    execution = execute_pipeline(trajectory, prompt, session["lm"])
                    execution["cached"] = False
                    reward = compute_reward(prompt, execution["output"], example["answer"], session["lm"], cfg)
                    if eval_cache is not None and cache_key is not None and execution["success"] and reward.get("judge_error") is None:
                        eval_cache.put(cache_key, execution, reward)
                append_journal(session, dict(kind="evaluation_attempt", split=split, episode=session["state"]["episode"],
                               question_id=example["id"], execution=execution, reward=reward,
                               cached=execution.get("cached", False),
                               resources=call_statistics(session["lm"].events[question_call_start:]),
                               timestamp=datetime.now(timezone.utc).isoformat()))
                pred = reward_prediction(execution["output"], cfg, task_type=example.get("task_type")) if "hotpotqa" == "mixed_routing" else reward_prediction(execution["output"], cfg)
                records.append(dict(question_id=example["id"], task_type=example.get("task_type", cfg["dataset"]),
                                    target=example["answer"], prediction=pred,
                                    trajectory=trajectory, execution=execution, **reward))
    finally:
        model.train(old_mode)
    exact = sum(record["exact_match"] for record in records)
    f1_mean = float(np.mean([record.get("answer_f1", 0.0) for record in records]))
    tip_rate = float(np.mean([bool(record.get("target_in_prediction", False)) for record in records]))
    scores = [record["judge_score"] for record in records if record["judge_score"] is not None]
    return dict(
        dataset=cfg["dataset"], run_profile=cfg["run_profile"], k=cfg["k"], seed=cfg["seed"], split=split,
        eval_decoding=cfg.get("eval_decoding", "greedy"),
        eval_cache_hits=cache_hits, eval_physical_llm_calls=len(session["lm"].events) - call_start,
        reward_mode=effective_reward_config(cfg)["reward_mode"],
        episode=session["state"]["episode"] if episode is None else episode,
        selected_checkpoint=session.get("loaded_checkpoint"), questions=len(records),
        question_ids=[r["question_id"] for r in records], exact_match_accuracy=exact / len(records),
        answer_f1=f1_mean,
        target_in_prediction_rate=tip_rate,
        semantic_fallback_count=sum(record["semantic_used"] for record in records),
        semantic_fallback_mean_score=float(np.mean(scores)) if scores else None,
        semantic_full_credit_count=sum(score == 1.0 for score in scores),
        mean_reward=float(np.mean([r["reward"] for r in records])),
        attempted_pipeline_evaluations=len(records),
        successful_pipeline_evaluations=sum(r["execution"]["success"] for r in records),
        failed_pipeline_evaluations=sum(not r["execution"]["success"] for r in records),
        cumulative_training_pipeline_evaluations=sum(r["attempted_pipeline_evaluations"] for r in session["state"]["train"]),
        seconds=time.perf_counter() - begin, records=records, **call_statistics(session["lm"].events[call_start:])
    )

def validate_selected(session):
    if "loaded_checkpoint" not in session:
        raise RuntimeError("Load intended checkpoint before standalone validation")
    result = evaluate_module(session, "validation")
    save_evaluation_artifact(session, result)
    refresh_test_summary(session["run_dir"])
    print(f"Validation EM: {result['exact_match_accuracy']*100:.1f}% | F1: {result['answer_f1']:.3f} | Substring: {result['target_in_prediction_rate']*100:.1f}% | Mean Reward (Semantic): {result['mean_reward']:.3f} ({result['mean_reward']*100:.1f}%)")
    return result


# 17. Final Testing
Final testing explicitly loads the validation-selected checkpoint and uses the fixed official-test subset. It never selects checkpoints. Full official-test evaluation is opt-in and can be expensive. Test artifacts do not mutate historical training checkpoints.

In [ ]:
def save_evaluation_artifact(session, result):
    path = session["run_dir"] / f"{result['split']}_{uuid.uuid4().hex[:8]}.json"
    write_json(path, result)
    return path

def test_model(session, full_official=False):
    if "loaded_checkpoint" not in session:
        raise RuntimeError("Final testing requires an explicitly loaded checkpoint")
    training_summary = json.loads((session["run_dir"] / "summary.json").read_text(encoding="utf-8"))
    if not training_summary["completed"]:
        raise RuntimeError("Finish training before final testing")
    best_path = session["run_dir"] / "best.json"
    if not best_path.exists():
        raise RuntimeError("No validation-selected checkpoint")
    best = json.loads(best_path.read_text(encoding="utf-8"))
    if Path(session["loaded_checkpoint"]).resolve() != Path(best["checkpoint"]).resolve():
        raise ValueError("Load validation-selected best checkpoint before final testing")
    if full_official:
        official = session["data"]["full_test"]
        examples = [dict(id=f"hotpotqa:test:{i}", split="test", index=i, question=item["question"], answer=item["answer"], input=format_hotpot_input(item["question"], item["context"]))
                    for i, item in enumerate(official)]
        result = evaluate_module(session, "full_test", examples=examples)
    else:
        result = evaluate_module(session, "test")
    result["checkpoint_selection_rule"] = "maximum validation exact match; earliest episode on ties"
    path = save_evaluation_artifact(session, result)
    refresh_test_summary(session["run_dir"])
    print(f"Held-out EM: {result['exact_match_accuracy']*100:.1f}% | F1: {result['answer_f1']:.3f} | Substring: {result['target_in_prediction_rate']*100:.1f}% | Mean Reward (Semantic): {result['mean_reward']:.3f} ({result['mean_reward']*100:.1f}%) | Cache hits: {result['eval_cache_hits']}/{result['questions']} | artifact: {path}")
    return result

def run_zero_rl_baselines(cfg):
    """Evaluate zero-RL DSPy-Predict, DSPy-CoT, and Random-Pipeline on the held-out test split."""
    session = initialize_experiment(cfg, create_run=False)
    try:
        out_dir = Path(cfg["results_root"]) / cfg["dataset"] / "zero_rl_baselines" / f"seed{cfg['seed']}"
        out_dir.mkdir(parents=True, exist_ok=True)
        rng = random.Random(cfg["seed"] + 999)
        def make_fixed(mod, sig):
            return lambda _ex: dict(actions=[mod, sig, "stop"], log_probs=[0.0, 0.0, 0.0], module=mod, signature=sig)
        def make_random(_ex):
            m, s = rng.choice(MODULES), rng.choice(SIGNATURES)
            return dict(actions=[m, s, "stop"], log_probs=[0.0, 0.0, 0.0], module=m, signature=s)
        results = {}
        for name, fn in [("dspy_predict", make_fixed("Predict", "question -> answer")),
                         ("dspy_cot", make_fixed("CoT", "question -> answer")),
                         ("random_pipeline", make_random)]:
            res = evaluate_module(session, "test", episode=0, policy_override_fn=fn)
            res.pop("records", None)
            results[name] = res
            print(f"[Zero-RL {name}] exact_match_accuracy={res['exact_match_accuracy']*100:.1f}% | Sem={res['mean_reward']*100:.1f}% | Cache={res['eval_cache_hits']}/{res['questions']}")
        payload = dict(dataset=cfg["dataset"], seed=cfg["seed"], test_size=cfg["test_size"],
                       ollama_model=cfg["ollama_model"], selection_id=session["data"]["selection_id"],
                       baselines=results, saved_at=datetime.now(timezone.utc).isoformat())
        out_path = out_dir / "zero_rl_baselines.json"
        write_json(out_path, payload, overwrite=True)
        if IN_COLAB and BACKUP_TO_DRIVE:
            exec_short = cfg["ollama_model"].replace(":", "-")
            policy_short = cfg["policy_model"].split("/")[-1].replace(":", "-")
            drive_dst = Path(f"/content/drive/MyDrive/APEX_GRPO_v3/{policy_short}_{exec_short}/{cfg['dataset']}/zero_rl_baselines/seed{cfg['seed']}/zero_rl_baselines.json")
            copy_backup_file(out_path, drive_dst)
            print("Zero-RL backup complete:", drive_dst)
        return payload
    finally:
        release_experiment(session)


# 18. Metrics and Statistics
JSONL holds episode and validation rows. `attempts.jsonl` records granular attempt-level reservations, LLM call timestamps, physical execution vs deduplication cache hits, and failure recovery. Extended APEX-GRPO metrics track:
- `k_ceiling`: Analytical pre-allocation ceiling computed from Pillar 1.
- `policy_entropy`: Policy action entropy across the 30 pipelines.
- `physical_lm_calls`: Number of physical HTTP calls dispatched to Ollama.
- `cached_pipeline_executions`: Redundant candidate calls avoided via Pillar 2 deduplication.
- `unique_pipelines_count` & `unique_pipeline_ratio`: Policy diversity within the group.
- `optimizer_step` & `skip_reason`: Audit log of Pillar 4 zero-gradient updates vs skips.


In [ ]:
def call_statistics(events):
    def total(key):
        return sum(event[key] for event in events) if all(event.get(key) is not None for event in events) else None
    return dict(execution_llm_calls=len(events), physical_http_requests=None, prompt_tokens=total("prompt_tokens"),
                generated_tokens=total("generated_tokens"), llm_seconds=sum(e["seconds"] for e in events),
                llm_failed_calls=sum(e["error"] is not None for e in events),
                call_count_definition="DSPy LM.forward attempts, including semantic judge and adapter retries")

def gpu_statistics(model):
    if not next(model.parameters()).is_cuda:
        return dict(peak_allocated_gpu_bytes=None, peak_reserved_gpu_bytes=None)
    return dict(peak_allocated_gpu_bytes=torch.cuda.max_memory_allocated(), peak_reserved_gpu_bytes=torch.cuda.max_memory_reserved())

def write_jsonl(path, rows):
    path = Path(path); temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
        stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)

def synchronize_logs(session):
    if session["run_dir"] is not None:
        write_jsonl(session["run_dir"] / "training.jsonl", session["state"]["train"])
        write_jsonl(session["run_dir"] / "validation.jsonl", session["state"]["validation"])

def write_summary(session):
    summary = experiment_summary(session)
    write_json(session["run_dir"] / "summary.json", summary, overwrite=True)
    result = refresh_test_summary(session["run_dir"])
    latest_path = session["run_dir"] / "latest.json"
    if latest_path.exists() and json.loads(latest_path.read_text(encoding="utf-8"))["episode"] == session["state"]["episode"]:
        backup_results(session["config"])
    return result

def refresh_test_summary(root):
    root = Path(root)
    summary = json.loads((root / "summary.json").read_text(encoding="utf-8"))
    tests = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("test_*.json"))]
    extra_validation = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("validation_*.json"))]
    full_tests = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("full_test_*.json"))]
    scores = [result["exact_match_accuracy"] for result in tests]
    summary["final_held_out_accuracy"] = scores[0] if scores and all(score == scores[0] for score in scores) else None
    f1_scores = [result.get("answer_f1") for result in tests]
    summary["final_held_out_f1"] = f1_scores[0] if f1_scores and all(score == f1_scores[0] for score in f1_scores) else None
    mr_scores = [result.get("mean_reward") for result in tests]
    summary["final_held_out_mean_reward"] = mr_scores[0] if mr_scores and all(score == mr_scores[0] for score in mr_scores) else None
    summary["test_status"] = "available" if summary["final_held_out_accuracy"] is not None else "repeated tests disagree" if scores else "not run"
    validation = read_jsonl(root / "validation.jsonl")
    evaluations = validation + tests + extra_validation + full_tests
    summary["evaluation_llm_calls"] = sum(row["execution_llm_calls"] for row in evaluations)
    summary["evaluation_pipeline_evaluations"] = sum(row["attempted_pipeline_evaluations"] for row in evaluations)
    summary["evaluation_seconds"] = sum(row["seconds"] for row in evaluations)
    summary["total_llm_calls_in_effective_history"] = summary["execution_llm_calls"] + summary["evaluation_llm_calls"]
    journal = read_jsonl(root / "attempts.jsonl") if (root / "attempts.jsonl").exists() else []
    attempts = [row for row in journal if row["kind"] in ("training_attempt", "evaluation_attempt")]
    summary["resource_journal_complete"] = all("resources" in row for row in attempts)
    write_json(root / "summary.json", summary, overwrite=True)
    return summary

def experiment_summary(session):
    state, cfg = session["state"], session["config"]; rows = state["train"]
    attempts = sum(r["attempted_pipeline_evaluations"] for r in rows)
    latest_path = session["run_dir"] / "latest.json"
    durable = json.loads(latest_path.read_text(encoding="utf-8"))["episode"] if latest_path.exists() else 0
    tests = []
    for path in sorted(session["run_dir"].glob("test_*.json")):
        tests.append(dict(path=str(path), **json.loads(path.read_text(encoding="utf-8"))))
    scores = [t["exact_match_accuracy"] for t in tests]
    final_accuracy = scores[0] if scores and all(score == scores[0] for score in scores) else None
    f1_scores = [t.get("answer_f1") for t in tests]
    final_f1 = f1_scores[0] if f1_scores and all(score == f1_scores[0] for score in f1_scores) else None
    test_status = "available" if scores else "not run"
    if scores and final_accuracy is None:
        test_status = "repeated tests disagree; inspect artifacts, no best-score selection"
    def total(key):
        values = [r[key] for r in rows]
        return sum(values) if all(v is not None for v in values) else None
    evaluation_calls = sum(v["execution_llm_calls"] for v in state["validation"]) + sum(t["execution_llm_calls"] for t in tests)
    return dict(training_candidates_reserved=reserved_training_candidates(session), candidate_budget=cfg["candidate_budget"],
                budget_exhausted=state.get("budget_exhausted", False), config=cfg, selection_id=session["data"]["selection_id"],
                initial_policy_digest=session["initial_policy_digest"],
                execution_model_digest=session["llm_metadata"].get("model", {}).get("digest"),
                runtime_identity=dict(packages=VERSIONS, ollama_version=session["llm_metadata"].get("version")), run_dir=str(session["run_dir"]),
                completed=(state["episode"] == cfg["num_episodes"] or state.get("budget_exhausted", False)) and durable == state["episode"] and any(row["episode"] == state["episode"] for row in state["validation"]),
                episode=state["episode"], durable_episode=durable, optimizer_updates=state["updates"],
                unique_training_questions=len(state["seen"]), best=state["best"], test_status=test_status,
                final_held_out_accuracy=final_accuracy, final_held_out_f1=final_f1,
                training_time=state["cumulative_training_seconds"],
                attempted_pipeline_evaluations=attempts, successful_pipeline_evaluations=total("successful_pipeline_evaluations"),
                failed_pipeline_evaluations=total("failed_pipeline_evaluations"), execution_llm_calls=total("execution_llm_calls"),
                physical_execution_llm_calls=total("physical_lm_calls"),
                cached_pipeline_executions=total("cached_pipeline_executions"),
                zero_variance_skipped_updates=sum(not r.get("optimizer_step", True) for r in rows),
                average_unique_pipeline_ratio=float(np.mean([r.get("unique_pipeline_ratio", 1.0) for r in rows])) if rows else None,
                prompt_tokens=total("prompt_tokens"), generated_tokens=total("generated_tokens"), physical_http_requests=None,
                training_mean_reward=float(np.mean([r["mean_group_reward"] for r in rows])) if rows else None,
                training_reward_variance=float(np.var([x for r in rows for x in r["rewards"]])) if rows else None,
                zero_advantage_group_percentage=100 * float(np.mean([r["zero_advantage_group"] for r in rows])) if rows else None,
                average_pipeline_execution_latency=sum(r["pipeline_execution_seconds"] for r in rows) / attempts if attempts else None)


# 19. Compare Adaptive-K and Static-K

After executing the definition cells, pass explicit completed run directories:

```python
runs = compare_runs([STATIC_RUN_DIR, ADAPTIVE_RUN_DIR],
                    comparison_axis="budget", candidate_cap=1000)
```

The result reports training attempts for each run; a common cap does not mean
both used the same number of candidates. Static journals record completed attempts,
so an in-flight interruption can be unmeasured. Validation and test calls are not
included in this cap. Comparisons check dataset, seed, model and runtime identities.
Both `bounded-rescue-v1` and `apex-adaptive-rollout-v2` artifact schemas are verified.


In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

def verify_adaptive_artifacts(root, summary):
    """Reject inconsistent histories before plotting; never rewrite source artifacts."""
    root = Path(root); cfg = summary["config"]
    train = read_jsonl(root / "training.jsonl"); validation = read_jsonl(root / "validation.jsonl")
    def require(condition, message):
        if not condition:
            raise ValueError(f"Inconsistent run artifacts at {root}: {message}")
    alloc_ver = cfg.get("allocation_version")
    require(alloc_ver in ("bounded-rescue-v1", "apex-adaptive-rollout-v2", "apex-adaptive-rollout-v3"), "unsupported allocation")
    final_episode = summary["episode"]
    require(final_episode > 0 and final_episode == summary["durable_episode"], "incomplete durable history")
    if final_episode != cfg["num_episodes"]:
        require(summary.get("budget_exhausted") is True and cfg.get("candidate_budget") is not None, "unjustified early stop")
        journal = read_jsonl(root / "attempts.jsonl")
        reservations = sum(row.get("kind") == "training_reservation" for row in journal)
        require(reservations == summary.get("training_candidates_reserved") and 0 <= cfg["candidate_budget"] - reservations < cfg["adaptive_levels"][0], "invalid budget exhaustion")
    require([row["episode"] for row in train] == list(range(1, final_episode + 1)), "missing/duplicate training episodes")
    cumulative = 0; seen = set()
    for row in train:
        require((row["dataset"], row["seed"]) == (cfg["dataset"], cfg["seed"]) and row["k"] in cfg["adaptive_levels"], "episode identity")
        require(row["attempted_pipeline_evaluations"] == row["k"], "candidate budget")
        require(row["successful_pipeline_evaluations"] + row["failed_pipeline_evaluations"] == row["k"], "candidate outcomes")
        require(len(row["rewards"]) == len(row["advantages"]) == len(row["actions"]) == len(row["signatures"]) == row["k"], "candidate records")
        mean, advantages = group_advantages(row["rewards"])
        require(np.isclose(row["mean_group_reward"], mean) and np.allclose(row["advantages"], advantages), "centered reward statistics")
        require(row["zero_advantage_group"] == bool(np.all(advantages == 0)), "zero-advantage flag")
        cumulative += row["k"]; seen.add(row["question_id"])
        require(row["unique_training_questions"] == len(seen), "unique question count")
        if alloc_ver == "apex-adaptive-rollout-v3":
            if cfg.get("use_zero_var_mask", True):
                spread = float(max(row["rewards"]) - min(row["rewards"]))
                expected_step = not (row["zero_advantage_group"] or spread <= cfg.get("reward_contrast", 0.0))
                require(row.get("optimizer_step") == expected_step, "v3 optimizer step masking consistency")
            else:
                require(row.get("optimizer_step") is True and row["optimizer_update_count"] == row["episode"], "unmasked update count")
        elif alloc_ver == "apex-adaptive-rollout-v2":
            require(row.get("optimizer_step") == (not row["zero_advantage_group"]), "optimizer step masking consistency")
        else:
            require(row["optimizer_update_count"] == row["episode"], "optimizer update count")
    require(summary["attempted_pipeline_evaluations"] == cumulative, "summary training budget")
    expected_updates = sum(r.get("optimizer_step", True) for r in train) if alloc_ver in ("apex-adaptive-rollout-v2", "apex-adaptive-rollout-v3") else len(train)
    require(summary["optimizer_updates"] == expected_updates and summary["unique_training_questions"] == len(seen), "summary updates/questions")
    for key in ("successful_pipeline_evaluations", "failed_pipeline_evaluations", "execution_llm_calls"):
        require(summary[key] == sum(row[key] for row in train), "summary " + key)
    require(np.isclose(summary["training_time"], sum(row["episode_seconds"] for row in train)), "summary training time")
    expected = [0] + [episode for episode in range(1, final_episode + 1)
                      if episode % cfg["validation_interval"] == 0 or episode == final_episode]
    require([row["episode"] for row in validation] == expected, "validation progression")
    validation_ids = validation[0]["question_ids"]
    require(len(validation_ids) == len(set(validation_ids)) == cfg["validation_size"], "validation question IDs")
    for row in validation:
        require(row["question_ids"] == validation_ids and not seen.intersection(row["question_ids"]), "validation identity/leakage")
        require(row["questions"] == row["attempted_pipeline_evaluations"] == cfg["validation_size"], "validation pipeline budget")
        require(row["cumulative_training_pipeline_evaluations"] == sum(t["k"] for t in train if t["episode"] <= row["episode"]), "validation training-budget provenance")
        require(row["successful_pipeline_evaluations"] + row["failed_pipeline_evaluations"] == row["questions"], "validation outcomes")
    metric = "numeric_exact_match_accuracy" if cfg["dataset"] == "gsm8k" else "exact_match_accuracy"
    selected = max(validation, key=lambda row: (row[metric], -row["episode"]))
    best = summary.get("best")
    require(best is not None and best["episode"] == selected["episode"] and np.isclose(best["score"], selected[metric]), "validation-only checkpoint selection")
    tests = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("test_*.json"))]
    for result in tests:
        require(result["split"] == "test" and result["questions"] == cfg["test_size"], "held-out test scope")
        require(Path(result["selected_checkpoint"]).resolve() == Path(best["checkpoint"]).resolve(), "held-out checkpoint provenance")
        require(len(result["question_ids"]) == len(set(result["question_ids"])) == cfg["test_size"], "held-out question IDs")
        require(not (seen | set(validation_ids)).intersection(result["question_ids"]), "held-out leakage")
        require(result["question_ids"] == tests[0]["question_ids"], "repeated test identity")
    scores = [row[metric] for row in tests]
    accuracy = scores[0] if scores and all(score == scores[0] for score in scores) else None
    require(summary["final_held_out_accuracy"] == accuracy, "held-out summary score")
    latest = json.loads((root / "latest.json").read_text(encoding="utf-8"))
    require(latest["episode"] == final_episode and Path(latest["checkpoint"]).is_file(), "latest durable checkpoint")
    require(json.loads((root / "best.json").read_text(encoding="utf-8")) == best and Path(best["checkpoint"]).is_file(), "selected checkpoint pointer")
    metadata_path = root / "config.json"
    require(metadata_path.is_file(), "missing configuration snapshot")
    return dict(train=train, validation=validation)

def compare_runs(run_dirs, comparison_axis="episodes", candidate_cap=None):
    if not run_dirs:
        raise ValueError("Provide explicit completed run directories")
    if comparison_axis not in ("episodes", "budget"):
        raise ValueError("comparison_axis must be episodes or budget")
    if comparison_axis == "budget" and (type(candidate_cap) is not int or candidate_cap <= 0):
        raise ValueError("Budget comparison requires a positive candidate_cap")
    runs = []; reference = None; seen = set()
    ignore = {
        "k", "protocol", "protocol_version", "results_root", "run_profile",
        "adaptive_levels", "rescue_mode", "reward_contrast", "failure_reward_threshold",
        "candidate_budget", "allocation_version", "use_adaptive_k", "use_prealloc",
        "use_without_replacement", "rescue_is_correction", "use_dedup_cache",
        "normalize_loss_by_k", "use_zero_var_mask", "debiased_logits",
        "entropy_coef", "grad_clip_norm", "eval_cache_enabled", "loss", "policy_sampling",
        "signature_action_token_ids", "lambda_penalty",
    }
    for root in map(Path, run_dirs):
        if root.resolve() in seen:
            raise ValueError("Duplicate run directory")
        seen.add(root.resolve())
        summary = json.loads((root / "summary.json").read_text(encoding="utf-8"))
        if not summary["completed"]:
            raise ValueError("Comparison requires complete runs")
        canonical = canonical_experiment_config(summary["config"])
        if comparison_axis == "budget":
            ignore = ignore | {"num_episodes"}
        cfg = {k: v for k, v in canonical["config"].items() if k not in ignore}
        identity = dict(config=cfg, reward=canonical["reward"], selection_id=summary["selection_id"],
                        initial_policy_digest=summary["initial_policy_digest"],
                        execution_model_digest=summary["execution_model_digest"])
        if reference is not None and identity != reference:
            raise ValueError("Incompatible data, seed, reward, or model controls across compared runs")
        reference = identity
        history = verify_adaptive_artifacts(root, summary)
        journal_path = root / "attempts.jsonl"
        if comparison_axis == "budget" and not journal_path.exists():
            raise ValueError("Budget comparison requires attempts.jsonl")
        journal = read_jsonl(journal_path) if journal_path.exists() else []
        measured = sum(row.get("kind") == "training_reservation" for row in journal) if journal else summary["attempted_pipeline_evaluations"]
        if comparison_axis == "budget" and (measured < summary["attempted_pipeline_evaluations"] or measured > candidate_cap):
            raise ValueError("Training attempts missing or exceed the candidate cap")
        runs.append(dict(summary=summary, comparison_axis=comparison_axis, candidate_cap=candidate_cap,
                         measured_training_attempts=measured, **history))
    return runs

def adaptive_results(session):
    rows = session["state"]["train"]
    return dict(
        episodes=len(rows),
        candidates=sum(r["attempted_pipeline_evaluations"] for r in rows),
        realized_k={k: sum(r["k"] == k for r in rows) for k in session["config"]["adaptive_levels"]},
        k_ceilings={k: sum(r.get("k_ceiling") == k for r in rows) for k in session["config"]["adaptive_levels"]},
        physical_calls=sum(r.get("physical_lm_calls", r["attempted_pipeline_evaluations"]) for r in rows),
        cached_executions=sum(r.get("cached_pipeline_executions", 0) for r in rows),
        optimizer_steps=sum(r.get("optimizer_step", True) for r in rows),
        zero_variance_skips=sum(r.get("skip_reason") == "zero_variance_group" for r in rows),
        below_contrast_skips=sum(r.get("skip_reason") == "below_contrast_threshold" for r in rows),
        rescue_samples=sum(r.get("rescue_samples_count", 0) for r in rows),
        stop_reasons={reason: sum(r.get("stop_reason") == reason for r in rows)
                      for reason in ("contrast", "all_correct_exit", "ceiling_reached", "maximum_k",
                                     "failure_guard", "non_failure_homogeneous", "budget_exhausted", "static_fixed_k")}
    )


# 20. Adaptive-K visualization


In [ ]:
def plot_adaptive_results(session):
    data = adaptive_results(session)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    levels = session["config"]["adaptive_levels"]
    labels = [str(k) for k in levels]
    x = np.arange(len(labels))
    width = 0.35 if len(levels) > 1 else 0.25
    axes[0].bar(x - width / 2, [data["realized_k"].get(k, 0) for k in levels], width, label="Realized K", color="tab:blue")
    axes[0].bar(x + width / 2, [data["k_ceilings"].get(k, 0) for k in levels], width, label="Ceiling (Pillar 1)", color="tab:cyan", alpha=0.7)
    axes[0].set(xlabel="Group Size K", ylabel="Episodes", title="Group Size vs Ceiling Allocation")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend()

    phys = data["physical_calls"]
    cached = data["cached_executions"]
    axes[1].bar(["Physical Ollama", "Cached (Deduplicated)"], [phys, cached], color=["tab:orange", "tab:green"])
    axes[1].set(ylabel="LM Evaluations", title=f"Deduplication Savings ({cached}/{max(1, phys + cached)} saved)")

    updates = data["optimizer_steps"]
    zv_skips = data["zero_variance_skips"]
    bc_skips = data["below_contrast_skips"]
    axes[2].bar(["Optimizer Step", "Zero-Var Skip", "Contrast Skip"], [updates, zv_skips, bc_skips],
                color=["tab:purple", "tab:gray", "tab:olive"])
    axes[2].set(ylabel="Episodes", title=f"Pillar 4 Masking ({zv_skips + bc_skips} skips)")
    fig.tight_layout()
    return fig


# 21. Reproducibility and Experiment Summary
Keep `RUN_OPERATION=False` when running definition cells. Rebuild CONFIG after edits and explicitly enable the operation below when ready. Training does not automatically test. `RUN_ALL_K=True` requests sequential independent training; each policy is released before the next.

Resume/validate/test require SELECTED_CHECKPOINT. For resume use latest.json; for final testing use best.json. Remote inference is not guaranteed deterministic. First-token collisions, last-number extraction, failure rewards and K-dependent gradient scaling remain documented limitations.

Notebook-local synthetic checks below use tiny test policies and mocks, never a substituted research model. They verify mechanics, never generate benchmark results. See [usage guide](STATIC_K_EXPERIMENT_GUIDE.md).

In [ ]:
def run_operation(cfg,mode,checkpoint=None,run_dirs=None,all_k=False):
    if mode=="compare":
        return compare_runs(run_dirs or [])
    if mode not in ("initialize","train","resume","validate","test"):
        raise ValueError("MODE must be initialize/train/resume/validate/test/compare")
    if all_k:
        raise ValueError("RUN_ALL_K is unavailable for adaptive runs")
    if all_k:
        if mode!="train":
            raise ValueError("RUN_ALL_K requires MODE=train")
        return run_all_k_sequential(cfg)
    if mode in ("resume","validate","test"):
        if checkpoint is None:
            raise ValueError("Set SELECTED_CHECKPOINT to an explicit .pt path")
        session=load_selected_experiment(checkpoint,cfg,for_resume=mode=="resume")
    else:
        session=initialize_experiment(cfg)
    if mode in ("train","resume"):
        train_grpo(session)
    elif mode=="validate":
        validate_selected(session)
    elif mode=="test":
        test_model(session)
    return session

COMPARISON_RUN_DIRS=[]
if RUN_OPERATION:
    CONFIG=build_config()
    if "SESSION" in globals() and isinstance(SESSION,dict) and "policy" in SESSION:
        release_experiment(SESSION)
    SESSION=run_operation(CONFIG,MODE,checkpoint=SELECTED_CHECKPOINT,
                          run_dirs=COMPARISON_RUN_DIRS,all_k=RUN_ALL_K)
else:
    print("Setup complete. No training/test launched. Run adaptive_self_checks() for synthetic controller checks.")

In [ ]:
def adaptive_self_checks():
    print("Running APEX-GRPO v3 self-checks...")
    # 1. Verify all 9 RUN_PROFILES build cleanly
    for pname in RUN_PROFILES:
        c = build_config(run_profile=pname, seed=42, preset="smoke")
        assert c["run_profile"] == pname
        assert resolve_drive_backup_dir(c).name == "seed42"
    
    raw = "The Scott Family Park!"
    assert normalize_answer(raw) == "scott family park"
    f1, prec, rec = answer_f1("Scott Family Park", "the scott family park.")
    assert np.isclose(f1, 1.0) and np.isclose(prec, 1.0) and np.isclose(rec, 1.0)

    # 2. Verify Pillar 1 Pre-Allocation on (2, 4, 8, 12)
    cfg = dict(adaptive_levels=[2, 4, 8, 12], use_adaptive_k=True, use_prealloc=True,
               rescue_mode="failure_only", reward_contrast=0.5, failure_reward_threshold=0.25, lambda_penalty=0.25)
    uniform_p = np.ones(30) / 30.0
    k_u, _, _, ent_u = solve_preallocation_ceiling(uniform_p, cfg)
    assert k_u == 12 and np.isclose(ent_u, math.log(30), atol=1e-4), f"Got k={k_u}, ent={ent_u}"

    p_mid = np.array([0.45, 0.20, 0.10] + [0.25 / 27.0] * 27)
    k_m, _, _, _ = solve_preallocation_ceiling(p_mid, cfg)
    assert k_m == 12

    p_conc = np.array([0.75, 0.10] + [0.15 / 28.0] * 28)
    k_conc, _, _, _ = solve_preallocation_ceiling(p_conc, cfg)
    assert k_conc == 4

    collapsed_p = np.zeros(30); collapsed_p[0] = 0.99; collapsed_p[1:] = 0.01 / 29.0
    k_c, _, _, _ = solve_preallocation_ceiling(collapsed_p, cfg)
    assert k_c == 2

    k_reg, _, _, _ = solve_preallocation_ceiling(uniform_p, cfg, historical_pass_rate=(0.0, 3))
    assert k_reg == 2

    # 3. Verify Pillar 3 Decision Gate (REWARD_CONTRAST=0.5, K_min=2)
    assert adaptive_decision([1.0, 1.0], False, cfg, k_ceiling=12) == "all_correct_exit"
    assert adaptive_decision([1.0, 0.0], False, cfg, k_ceiling=12) == "contrast"
    assert adaptive_decision([0.8, 0.0], False, cfg, k_ceiling=12) == "contrast"
    assert adaptive_decision([0.8, 0.9], False, cfg, k_ceiling=12) == "non_failure_homogeneous"
    assert adaptive_decision([0.5, 0.5], False, cfg, k_ceiling=12) == "non_failure_homogeneous"
    assert adaptive_decision([0.0, 0.0], False, cfg, k_ceiling=2) == "ceiling_reached"
    assert adaptive_decision([0.0, 0.0], False, cfg, k_ceiling=8) is None
    assert adaptive_decision([0.0] * 4, False, cfg, k_ceiling=8) is None
    assert adaptive_decision([0.0] * 8, False, cfg, k_ceiling=8) == "ceiling_reached"
    assert adaptive_decision([0.0, 0.0], True, cfg, k_ceiling=12) == "failure_guard"

    static_cfg = dict(cfg, adaptive_levels=[5], use_adaptive_k=False)
    assert adaptive_decision([0.0] * 5, False, static_cfg, k_ceiling=5) == "static_fixed_k"

    # 4. Verify 1/K Normalized & IS-Weighted Loss
    mean, adv = group_advantages([0, 0, 1])
    assert np.allclose(adv, [-1 / 3, -1 / 3, 2 / 3])
    dummy_lp = [[torch.tensor(-1.0)], [torch.tensor(-2.0)], [torch.tensor(-3.0)]]
    loss_val = figure2_loss(dummy_lp, adv, normalize_by_k=True, weights=[1.0, 1.0, 1.0])
    assert np.isclose(float(loss_val), 1.0 / 3.0)
    loss_raw = figure2_loss(dummy_lp, adv, normalize_by_k=False, weights=[1.0, 1.0, 1.0])
    assert np.isclose(float(loss_raw), 1.0)
    loss_is = figure2_loss(dummy_lp, adv, normalize_by_k=True, weights=[0.5, 0.5, 0.5])
    assert np.isclose(float(loss_is), 1.0 / 6.0)

    print("All APEX-GRPO v3 self-checks passed successfully!")


## Explicit operations

Set one switch to `True` and run its cell. Reset it to `False` afterward. A baseline run has 200 episodes, validation at 0/50/100/150/200, and checkpoints every 10 episodes. A finite candidate budget can stop earlier; no episode-200 checkpoint then exists. Test evaluations use the same held-out subset and should be reported separately; do not select a checkpoint using test scores.

In [ ]:
# Optional Zero-RL Baselines (DSPy-Predict, DSPy-CoT, Random-Pipeline) on the held-out test split.
RUN_ZERO_RL_BASELINES = False
if RUN_ZERO_RL_BASELINES:
    CONFIG = build_config()
    ZERO_RL_RESULTS = run_zero_rl_baselines(CONFIG)
else:
    print("Zero-RL baseline evaluation disabled. Set RUN_ZERO_RL_BASELINES = True to run.")


In [ ]:
# Fresh run. This can take a long time; it does not run until enabled.
START_NEW_TRAINING = False
if START_NEW_TRAINING:
    if "SESSION" in globals() and isinstance(SESSION, dict) and "policy" in SESSION:
        release_experiment(SESSION)
    CONFIG = build_config()
    SESSION = run_operation(CONFIG, "initialize")
    RUN_DIR = Path(SESSION["run_dir"])
    print("Run directory:", RUN_DIR)
    train_grpo(SESSION)
else:
    print("Fresh training disabled. Set START_NEW_TRAINING = True to start.")

In [ ]:
# Resume the latest durable checkpoint of an interrupted run.
RESUME_TRAINING = False
RUN_DIR = globals().get("RUN_DIR", None)  # Or set Path("/content/adaptive_grpo_results/.../run-id")
if RESUME_TRAINING:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR to the restored run directory")
    RUN_DIR = Path(RUN_DIR)
    CONFIG = json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8"))["config"]
    checkpoint = json.loads((RUN_DIR / "latest.json").read_text(encoding="utf-8"))["checkpoint"]
    if "SESSION" in globals() and isinstance(SESSION, dict) and "policy" in SESSION:
        release_experiment(SESSION)
    SESSION = load_selected_experiment(checkpoint, CONFIG, for_resume=True)
    print("Resuming run directory:", SESSION["run_dir"])
    train_grpo(SESSION)
else:
    print("Resume disabled.")


In [ ]:
# Read validation results without running inference.
if RUN_DIR is not None:
    from IPython.display import display
    RUN_DIR = Path(RUN_DIR)
    validation_results = read_jsonl(RUN_DIR / "validation.jsonl")
    display([{key: row.get(key) for key in ("episode", "questions", "exact_match_accuracy", "answer_f1", "target_in_prediction_rate", "mean_reward", "routing_tv_distance", "failed_pipeline_evaluations") if key in row} for row in validation_results])
    print("Best checkpoint:")
    display(json.loads((RUN_DIR / "best.json").read_text(encoding="utf-8")))
else:
    print("Set RUN_DIR to inspect a run.")


In [ ]:
def load_for_checkpoint_evaluation(run_dir, checkpoint):
    global SESSION
    run_dir = Path(run_dir)
    cfg = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))["config"]
    if "SESSION" in globals() and isinstance(SESSION, dict) and "policy" in SESSION:
        release_experiment(SESSION)
    SESSION = load_selected_experiment(checkpoint, cfg, for_resume=False)
    return SESSION

TEST_BEST = False
if TEST_BEST:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR first")
    RUN_DIR = Path(RUN_DIR)
    best = json.loads((RUN_DIR / "best.json").read_text(encoding="utf-8"))
    best_session = load_for_checkpoint_evaluation(RUN_DIR, best["checkpoint"])
    BEST_TEST = test_model(best_session)
    backup_results(best_session["config"])
    print(f"Best checkpoint held-out EM: {BEST_TEST['exact_match_accuracy']*100:.1f}% | F1: {BEST_TEST['answer_f1']:.3f} | Mean Reward (Semantic): {BEST_TEST['mean_reward']:.3f} ({BEST_TEST['mean_reward']*100:.1f}%)")
else:
    print("Best-checkpoint test disabled.")


In [ ]:
# Optional separate test of the fixed episode-200 checkpoint.
TEST_EPISODE_200 = False
if TEST_EPISODE_200:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR first")
    RUN_DIR = Path(RUN_DIR)
    latest = json.loads((RUN_DIR / "latest.json").read_text(encoding="utf-8"))
    summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
    if not summary["completed"] or latest["episode"] != summary["config"]["num_episodes"]:
        raise ValueError("This operation requires a completed run at its final episode")
    final_session = load_for_checkpoint_evaluation(RUN_DIR, latest["checkpoint"])
    EPISODE_200_TEST = evaluate_module(final_session, "test")
    EPISODE_200_TEST["checkpoint_selection_rule"] = "fixed episode 200; not selected by held-out results"
    final_test_path = RUN_DIR / f"episode_200_test_{uuid.uuid4().hex[:8]}.json"
    write_json(final_test_path, EPISODE_200_TEST)
    backup_results(final_session["config"])
    print(f"Episode-200 held-out EM: {EPISODE_200_TEST['exact_match_accuracy']*100:.1f}% | F1: {EPISODE_200_TEST['answer_f1']:.3f} | Mean Reward (Semantic): {EPISODE_200_TEST['mean_reward']:.3f} ({EPISODE_200_TEST['mean_reward']*100:.1f}%) | artifact: {final_test_path}")
else:
    print("Episode-200 test disabled.")


### Plots for the best and episode-200 checkpoints

These figures read saved logs only. Each chart shows the complete recorded training history and marks its selected checkpoint; a held-out score is shown only if the corresponding explicit test has been run. The episode-200 figure requires an actual latest episode-200 checkpoint. Files are written under a new timestamped run-local `plots/` directory and backed up to Drive.

In [ ]:
def plot_checkpoint_figures(run_dir):
    run_dir = Path(run_dir)
    train = read_jsonl(run_dir / "training.jsonl")
    validation = read_jsonl(run_dir / "validation.jsonl")
    if not train or not validation:
        raise ValueError("Training and validation records are required")
    summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
    best = json.loads((run_dir / "best.json").read_text(encoding="utf-8"))
    latest = json.loads((run_dir / "latest.json").read_text(encoding="utf-8"))
    checkpoints = [("best_checkpoint", best)]
    if latest["episode"] == 200 and Path(latest["checkpoint"]).is_file():
        checkpoints.append(("episode_200", latest))
    else:
        print("Episode-200 figure skipped: no latest episode-200 checkpoint.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    output_dir = run_dir / "plots" / stamp
    output_dir.mkdir(parents=True, exist_ok=False)
    episodes = [row["episode"] for row in train]
    for label, pointer in checkpoints:
        checkpoint = Path(pointer["checkpoint"])
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        pattern = "test_*.json" if label == "best_checkpoint" else "episode_200_test_*.json"
        tests = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(run_dir.glob(pattern))]
        tests = [t for t in tests if t.get("split") == "test" and t.get("episode") == pointer["episode"]
                 and t.get("selected_checkpoint") and Path(t["selected_checkpoint"]).resolve() == checkpoint.resolve()
                 and t.get("questions") == summary["config"]["test_size"]]
        if len({t["exact_match_accuracy"] for t in tests}) > 1:
            raise ValueError(f"Repeated {label} tests disagree; inspect saved test artifacts")
        fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
        ax = axes.flat
        ax[0].plot([v["episode"] for v in validation], [100*v["exact_match_accuracy"] for v in validation], marker="o", label="Exact Match (%)")
        if any("answer_f1" in v for v in validation):
            ax[0].plot([v["episode"] for v in validation], [100*v["answer_f1"] for v in validation], marker="^", linestyle=":", label="Answer F1 (%)")
        if any("mean_reward" in v for v in validation):
            ax[0].plot([v["episode"] for v in validation], [100*v["mean_reward"] for v in validation], marker="s", linestyle="--", label="Mean Reward (Semantic %)")
        ax[0].set(title="Validation Performance", xlabel="Episode", ylabel="Score (%)", ylim=(0, 100))
        ax[0].legend(loc="best", fontsize="small")
        ax[1].plot(episodes, [r["mean_group_reward"] for r in train])
        ax[1].set(title="Mean candidate reward", xlabel="Episode", ylabel="Reward")
        ax[2].plot(episodes, [r["gradient_norm"] for r in train])
        ax[2].set(title="Gradient norm", xlabel="Episode", ylabel="L2 norm")
        ax[3].step(episodes, [r["k"] for r in train], where="mid")
        ax[3].set(title="Realized adaptive group size", xlabel="Episode", ylabel="K")
        ax[4].plot(episodes, np.cumsum([r["attempted_pipeline_evaluations"] for r in train]))
        ax[4].set(title="Cumulative training pipeline evaluations", xlabel="Episode", ylabel="Pipeline evaluations")
        if tests:
            test_em = 100 * tests[0]["exact_match_accuracy"]
            test_f1 = 100 * tests[0].get("answer_f1", test_em)
            test_mr = 100 * tests[0].get("mean_reward", test_em)
            x_pos = np.arange(3)
            bars = ax[5].bar(x_pos, [test_em, test_f1, test_mr], color=["tab:blue", "tab:purple", "tab:orange"], width=0.5)
            ax[5].set_xticks(x_pos)
            ax[5].set_xticklabels(["Exact Match", "Answer F1", "Mean Reward\n(Semantic)"])
            ax[5].set(title=f"{label}: test (n={tests[0]['questions']})", ylabel="Score (%)", ylim=(0, 100))
            for b, val in zip(bars, [test_em, test_f1, test_mr]):
                ax[5].text(b.get_x() + b.get_width()/2, val + 1.5, f"{val:.1f}%", ha="center", fontsize="small", fontweight="bold")
        else:
            ax[5].axis("off")
            ax[5].text(.5, .5, "Held-out test not run", ha="center", va="center", transform=ax[5].transAxes)
        for a in ax[:5]:
            a.axvline(pointer["episode"], color="tab:red", linestyle="--", label=f"Checkpoint: episode {pointer['episode']}")
            a.legend(loc="best", fontsize="small")
        fig.suptitle(f"HotpotQA APEX-GRPO v3 | {label} | seed {summary['config']['seed']} | full recorded training history")
        for extension in ("png", "pdf"):
            fig.savefig(output_dir / f"{label}.{extension}", dpi=200, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    backup_results(summary["config"])
    print("Figures:", output_dir)
    return output_dir

GENERATE_PLOTS = False
if GENERATE_PLOTS:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR first")
    PLOT_DIR = plot_checkpoint_figures(RUN_DIR)
else:
    print("Plot generation disabled.")
